# 05 — Robustesse batch 3 et revue pré-batch4

Ce notebook applique le **contrat enfant `8tracks_v5_notebook05_v4`** sans modifier le protocole global `8tracks_v5`.

Principes scientifiques :

- `model_id` est l’identité scientifique du modèle ;
- `(model_id, random_state)` est une répétition d’exécution ; les seeds ne sont jamais classées ni sélectionnées individuellement ;
- le Pareto officiel de validation est calculé **séparément dans E1 … E8**, uniquement sur le panel 04C commun ;
- les seeds supplémentaires sont un **stress-test post-Pareto** réservé aux modèles stochastiques ;
- la politique de seuil 03B (`lower_quantile`, `upper_quantile`, `vote_threshold`) reste figée ; seuls les seuils numériques sont rematérialisés pour chaque nouvelle seed à partir des OOF batches 1–2 ;
- les tests de sensibilité aux seuils, à la composition du batch 3, aux folds de calibration, au front Pareto et au verrou spatial 03C sont **supporting-only** et ne peuvent pas modifier l’éligibilité ;
- les ablations sont exactes, intra-track et évaluées par différences appariées sur les seeds communes ;
- le batch 4 n’est jamais chargé ;
- aucune sélection finale, aucun score pondéré et aucun classement inter-track ne sont réalisés ici ;
- `pure_test_candidate_registry.parquet` et le verrou 05 constituent la frontière auditée transmise à 06A/06B et au notebook d’audit.


## A — Initialisation


In [20]:
from __future__ import annotations

import json
import platform
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

CURRENT_DIR = Path.cwd().resolve()
if (CURRENT_DIR / "src").is_dir():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "src").is_dir():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    raise RuntimeError(
        "Launch the notebook from the project root or from notebooks/."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 100)

from src import experiment_config as expcfg
from src.io.database_h5 import load_nir_uco_h5
from src.protocol_governance import (
    sha256_dataframe,
    sha256_file,
    sha256_payload,
    verify_frozen_protocol,
    resolve_protocol_for_execution,
)
from src.spectra.band_selection import select_wavelength_range_from_database
from src.utils import load_parquet
from src.workflows.simca_tables import (
    build_schema_manifest,
    read_simca_table,
    write_simca_table,
)
from src.workflows.simca_calibration_registry import (
    build_validation_execution_registry,
    stochastic_model_mask,
    validate_internal_calibration_manifest,
)
from src.workflows.simca_calibration_selection import (
    materialize_fixed_threshold_policy_for_runs,
)
from src.workflows.simca_internal_calibration import (
    run_internal_calibration_8tracks,
)
from src.workflows.simca import (
    run_locked_simca_validation_refit,
    run_locked_simca_validation_refit_checkpointed,
)
from src.workflows.simca_grid_evaluation import (
    build_validation_guardrails,
    evaluate_locked_validation_predictions,
)
from src.workflows.spatial_postprocessing_calibration import (
    build_locked_spatial_validation_outputs,
    verify_spatial_postprocessing_lock,
)
from src.workflows.simca_robustness import (
    assert_supporting_only,
    build_ablation_coverage,
    build_ablation_diagnostics,
    build_calibration_fold_sensitivity,
    build_descriptive_uncertainty_envelope,
    build_pareto_diagnostics,
    build_pareto_front_robustness,
    build_risk_coverage_curves,
    build_robustness_ablation_plan,
    build_robustness_review_guardrails,
    build_robustness_seed_execution_registry,
    build_seed_metrics,
    build_selection_unit_metrics,
    build_source_image_influence_diagnostics,
    build_spatial_sensitivity_plan,
    build_threshold_sensitivity_plan,
    build_threshold_stability_diagnostics,
    compute_seed_decision_disagreement,
    evaluate_calibration_fold_sensitivity,
    evaluate_spatial_sensitivity,
    evaluate_threshold_sensitivity,
    hash_robustness_contract,
    robustness_contract_payload,
    summarize_random_state_stability_metrics,
    validate_robustness_inputs,
)

print("Python:", platform.python_version())
print("PROJECT_ROOT:", PROJECT_ROOT)
%load_ext autoreload
%autoreload 2


Python: 3.14.6
PROJECT_ROOT: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## B — Contrat 05, chemins et frontières de données


In [2]:
if str(expcfg.PROTOCOL_VERSION) != "8tracks_v5":
    raise RuntimeError(
        f"Notebook 05 expects 8tracks_v5, got {expcfg.PROTOCOL_VERSION!r}."
    )
if str(expcfg.SIMCA_ROBUSTNESS_CONTRACT_VERSION) != "8tracks_v5_notebook05_v3":
    raise RuntimeError("Unexpected notebook-05 child-contract version.")
if bool(expcfg.SIMCA_ROBUSTNESS_ALLOW_BATCH4_INPUTS):
    raise RuntimeError("Notebook 05 must not allow batch-4 inputs.")
if bool(expcfg.SIMCA_ROBUSTNESS_FINAL_MODEL_SELECTION_PERFORMED):
    raise RuntimeError("Final model selection must remain downstream of notebook 05.")
if bool(expcfg.SIMCA_ROBUSTNESS_ALLOW_CROSS_TRACK_SELECTION):
    raise RuntimeError("Cross-track selection is forbidden in notebook 05.")
if bool(expcfg.SIMCA_ROBUSTNESS_RECOMPUTE_PARETO_AFTER_ADDITIONAL_SEEDS):
    raise RuntimeError("Pareto must remain frozen on the common 04C base panel.")
if bool(expcfg.SIMCA_ROBUSTNESS_RESELECT_THRESHOLD_POLICY_FOR_ADDITIONAL_SEEDS):
    raise RuntimeError("Additional seeds must not trigger threshold-policy reselection.")
if not bool(expcfg.SIMCA_ROBUSTNESS_RECALIBRATE_NUMERIC_THRESHOLDS_FOR_ADDITIONAL_SEEDS):
    raise RuntimeError("New seeds require seed-specific numeric threshold calibration.")

expected_tracks = {f"E{i}" for i in range(1, 9)}
configured_tracks = set(map(str, expcfg.SIMCA_ROBUSTNESS_TRACK_IDS))
if configured_tracks != expected_tracks:
    raise RuntimeError(
        f"Notebook 05 requires exactly E1-E8, got {sorted(configured_tracks)}."
    )

for track_id in sorted(expected_tracks):
    blocking_metrics = set(
        map(
            str,
            expcfg.SIMCA_ROBUSTNESS_BLOCKING_STABILITY_METRICS_BY_TRACK[
                track_id
            ],
        )
    )
    official_minimize = set(
        map(
            str,
            expcfg.SIMCA_ROBUSTNESS_PARETO_OBJECTIVES[track_id]["minimize"],
        )
    )
    if not blocking_metrics.issubset(official_minimize):
        raise RuntimeError(
            f"{track_id}: blocking stability metrics must be a subset of "
            "the official minimized Pareto objectives."
        )

USE_WAVELENGTH_WINDOW = bool(expcfg.USE_WAVELENGTH_WINDOW)
RESULTS_TAG = (
    f"{int(expcfg.WAVELENGTH_WINDOW_MIN_NM)}_"
    f"{int(expcfg.WAVELENGTH_WINDOW_MAX_NM)}"
    if USE_WAVELENGTH_WINDOW
    else expcfg.DEFAULT_RESULTS_TAG
)

PROTOCOL_DIR = PROJECT_ROOT.joinpath(*expcfg.PROTOCOL_ARTIFACT_RELATIVE_DIR)
DB_H5_PATH = PROJECT_ROOT.joinpath(*expcfg.DATABASE_H5_RELATIVE_PATH)

DATABASE_DIR = PROJECT_ROOT.joinpath(*expcfg.DATABASE_RESULTS_RELATIVE_DIR)
QC_DIR = PROJECT_ROOT.joinpath(*expcfg.QC_RESULTS_RELATIVE_DIR)
MATRIX_DIR = PROJECT_ROOT / "results" / f"{expcfg.MATRIX_RESULTS_DIR_PREFIX}_{RESULTS_TAG}"
PCA_DIR = PROJECT_ROOT / "results" / f"{expcfg.PCA_RESULTS_DIR_PREFIX}_{RESULTS_TAG}"
CALIBRATION_DIR = PROJECT_ROOT / "results" / f"{expcfg.INTERNAL_CALIBRATION_RESULTS_DIR_PREFIX}_{RESULTS_TAG}"
DOMAIN_DIR = PROJECT_ROOT / "results" / f"{expcfg.DOMAIN_SPATIAL_CALIBRATION_RESULTS_DIR_PREFIX}_{RESULTS_TAG}"
GRID_DIR = PROJECT_ROOT / "results" / f"{expcfg.SIMCA_GRID_SEARCH_RESULTS_DIR_PREFIX}_{RESULTS_TAG}"
OPTUNA_DIR = PROJECT_ROOT / "results" / f"{expcfg.SIMCA_OPTUNA_RESULTS_DIR_PREFIX}_{RESULTS_TAG}"
VALIDATION_DIR = PROJECT_ROOT / "results" / f"{expcfg.SIMCA_CONCAT_REFIT_RESULTS_DIR_PREFIX}_{RESULTS_TAG}"
OUTPUT_DIR = PROJECT_ROOT / "results" / f"{expcfg.SIMCA_ROBUSTNESS_RESULTS_DIR_PREFIX}_{RESULTS_TAG}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATHS = {
    key: OUTPUT_DIR / filename
    for key, filename in expcfg.SIMCA_ROBUSTNESS_OUTPUT_FILENAMES.items()
}

CHECKPOINT_ROOT = OUTPUT_DIR / "_checkpoints"
SEED_CALIBRATION_CHECKPOINT_DIR = CHECKPOINT_ROOT / "additional_seed_calibration"
SEED_VALIDATION_CHECKPOINT_DIR = CHECKPOINT_ROOT / "additional_seed_validation"
FOLD_SENSITIVITY_CHECKPOINT_DIR = CHECKPOINT_ROOT / "calibration_fold_sensitivity"

INPUT_03B = {
    key: CALIBRATION_DIR / expcfg.INTERNAL_CALIBRATION_OUTPUT_FILENAMES[key]
    for key in (
        "track_contracts",
        "folds",
        "model_catalog",
        "selected_models",
        "selected_runs",
        "selected_thresholds",
    )
}
INPUT_03B_MANIFEST = CALIBRATION_DIR / expcfg.INTERNAL_CALIBRATION_OUTPUT_FILENAMES["checkpoint_manifest"]

INPUT_03C = {
    key: DOMAIN_DIR / expcfg.DOMAIN_SPATIAL_CALIBRATION_OUTPUT_FILENAMES[key]
    for key in (
        "projection_eligibility",
        "spatial_calibration_metrics",
        "fragment_size_classes",
        "spatial_postprocessing_lock",
        "audit_manifest",
    )
}
INPUT_04A = {
    key: GRID_DIR / expcfg.SIMCA_GRID_SEARCH_OUTPUT_FILENAMES[key]
    for key in ("model_reference", "audit_manifest")
}
INPUT_04B_AUDIT = OPTUNA_DIR / expcfg.SIMCA_OPTUNA_OUTPUT_FILENAMES["audit_manifest"]
INPUT_04C = {
    key: VALIDATION_DIR / expcfg.SIMCA_CONCAT_REFIT_OUTPUT_FILENAMES[key]
    for key in (
        "object_predictions",
        "pixel_predictions",
        "metrics",
        "spatial_component_metrics",
        "guardrails",
        "technical_events",
        "protocol",
    )
}

# Earlier-stage files are lineage anchors only. No 01B test-ground-truth table is
# opened here: notebook 05 remains strictly pre-batch4.
EARLY_LINEAGE_PATHS = {
    "00.database_manifest": DATABASE_DIR / expcfg.DATABASE_OUTPUT_FILENAMES["manifest"],
    "01.qc_protocol": QC_DIR / expcfg.QC_OUTPUT_FILENAMES["protocol"],
    "01.split_manifest": QC_DIR / expcfg.QC_OUTPUT_FILENAMES["split_manifest"],
    "02.wavelength_config": MATRIX_DIR / expcfg.MATRIX_OUTPUT_FILENAMES["wavelength_config"],
    "02.preprocessing_validation": MATRIX_DIR / expcfg.MATRIX_OUTPUT_FILENAMES["preprocessing_validation"],
    "03.pca_selected": PCA_DIR / expcfg.PCA_OUTPUT_FILENAMES["selected"],
    "03.pca_selection_audit": PCA_DIR / expcfg.PCA_OUTPUT_FILENAMES["selection_audit"],
}

required_inputs = [
    *EARLY_LINEAGE_PATHS.values(),
    INPUT_03B_MANIFEST,
    *INPUT_03B.values(),
    *INPUT_03C.values(),
    *INPUT_04A.values(),
    *INPUT_04C.values(),
]
missing = [str(path) for path in required_inputs if not path.is_file()]
if missing:
    raise FileNotFoundError("Missing upstream notebook outputs:\n" + "\n".join(missing))

# protocol_checks_df = verify_frozen_protocol(PROTOCOL_DIR, strict=True)
# PROTOCOL_LOCK_PATH = PROTOCOL_DIR / expcfg.PROTOCOL_OUTPUT_FILENAMES["lock"]
# protocol_lock = json.loads(PROTOCOL_LOCK_PATH.read_text(encoding="utf-8"))
# PROTOCOL_HASH = str(protocol_lock["lock_sha256"])
#ROBUSTNESS_CONTRACT_HASH = hash_robustness_contract()
# ------------------------------------------------------------------
# Frozen parent lineage + mutable notebook-05 child contract
# ------------------------------------------------------------------

# ALLOW_PARENT_PROTOCOL_REUSE = True

# if ALLOW_PARENT_PROTOCOL_REUSE:
#     parent_manifest = json.loads(
#         INPUT_03B_MANIFEST.read_text(encoding="utf-8")
#     )

#     PROTOCOL_HASH = str(
#         parent_manifest["protocol_hash"]
#     )

#     protocol_checks_df = pd.DataFrame(
#         [
#             {
#                 "check": "parent_protocol_hash",
#                 "status": "pinned_to_03B_lineage",
#                 "value": PROTOCOL_HASH,
#                 "strict_current_config_match": False,
#                 "reason": (
#                     "Notebook-05 child-contract rerun while preserving "
#                     "the frozen 03B/03C/04A/04C parent lineage."
#                 ),
#             }
#         ]
#     )

# else:
#     protocol_checks_df = verify_frozen_protocol(
#         PROTOCOL_DIR,
#         strict=True,
#     )

#     PROTOCOL_LOCK_PATH = (
#         PROTOCOL_DIR
#         / expcfg.PROTOCOL_OUTPUT_FILENAMES["lock"]
#     )

#     protocol_lock = json.loads(
#         PROTOCOL_LOCK_PATH.read_text(encoding="utf-8")
#     )

#     PROTOCOL_HASH = str(
#         protocol_lock["lock_sha256"]
#     )

# # This one MUST use the current notebook-05 configuration.
# ROBUSTNESS_CONTRACT_HASH = hash_robustness_contract()
protocol_checks_df, PROTOCOL_HASH = (
    resolve_protocol_for_execution(
        PROTOCOL_DIR,
        upstream_manifest_path=INPUT_03B_MANIFEST,
    )
)
ROBUSTNESS_CONTRACT_HASH = hash_robustness_contract()

print("Parent protocol hash:", PROTOCOL_HASH)
print(
    "Notebook-05 child-contract hash:",
    ROBUSTNESS_CONTRACT_HASH,
)

print("Results tag:", RESULTS_TAG)
print("Global protocol hash:", PROTOCOL_HASH)
print("Notebook-05 contract hash:", ROBUSTNESS_CONTRACT_HASH)
print("Output:", OUTPUT_DIR)


Parent protocol hash: af19c5f35d24a81a9c34d0a37c1f2776a31f5aecc70a9bfc1eecac42f8f91fa2
Notebook-05 child-contract hash: 8084ad47efb798402bec55eb8999209803455a3262e844e655c022504ee834a6
Results tag: px_qc_v1
Global protocol hash: af19c5f35d24a81a9c34d0a37c1f2776a31f5aecc70a9bfc1eecac42f8f91fa2
Notebook-05 contract hash: 8084ad47efb798402bec55eb8999209803455a3262e844e655c022504ee834a6
Output: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\05_simca_validation_robustness_8tracks_v5_px_qc_v1


## C — Vérification de provenance 00 → 04C

Les sorties 03B–04C sont les entrées scientifiques effectives. Les sorties 00–03 servent d’ancrage de lignée. 04B est vérifié uniquement comme **audit de couverture** : aucune de ses tables de recherche n’entre dans la population scientifique de 05.


In [3]:
manifest_03b = json.loads(INPUT_03B_MANIFEST.read_text(encoding="utf-8"))
validate_internal_calibration_manifest(
    manifest_03b,
    INPUT_03B,
    required_artifacts=tuple(INPUT_03B),
    protocol_hash=PROTOCOL_HASH,
)

track_contracts_df = load_parquet(INPUT_03B["track_contracts"])
folds_df = load_parquet(INPUT_03B["folds"])
model_catalog_df = load_parquet(INPUT_03B["model_catalog"])
selected_models_df = load_parquet(INPUT_03B["selected_models"])
selected_runs_df = load_parquet(INPUT_03B["selected_runs"])
selected_threshold_rows_df = load_parquet(INPUT_03B["selected_thresholds"])

manifest_03c = json.loads(INPUT_03C["audit_manifest"].read_text(encoding="utf-8"))
if str(manifest_03c.get("protocol_hash")) != PROTOCOL_HASH:
    raise RuntimeError("03C and the frozen protocol have different hashes.")
if manifest_03c.get("input_03b_manifest_sha256") != sha256_file(INPUT_03B_MANIFEST):
    raise RuntimeError("03C was not built from the current 03B manifest.")
if manifest_03c.get("natural_execution_key") != ["model_id", "random_state"]:
    raise RuntimeError("03C does not expose the canonical execution key.")

projection_eligibility_df = load_parquet(INPUT_03C["projection_eligibility"])
spatial_calibration_metrics_df = load_parquet(INPUT_03C["spatial_calibration_metrics"])
fragment_size_classes_df = load_parquet(INPUT_03C["fragment_size_classes"])
spatial_lock = json.loads(INPUT_03C["spatial_postprocessing_lock"].read_text(encoding="utf-8"))
verify_spatial_postprocessing_lock(
    spatial_lock,
    spatial_calibration_metrics_df,
    fragment_size_classes_df,
)

for key in ("projection_eligibility", "spatial_calibration_metrics", "fragment_size_classes"):
    expected = str(manifest_03c["output_artifacts"][key]["sha256"])
    if sha256_file(INPUT_03C[key]) != expected:
        raise RuntimeError(f"03C hash mismatch for {key}.")
if sha256_file(INPUT_03C["spatial_postprocessing_lock"]) != str(
    manifest_03c["spatial_postprocessing_lock_sha256"]
):
    raise RuntimeError("03C spatial-lock hash mismatch.")

manifest_04a = json.loads(INPUT_04A["audit_manifest"].read_text(encoding="utf-8"))
if str(manifest_04a.get("protocol_hash")) != PROTOCOL_HASH:
    raise RuntimeError("04A and the frozen protocol have different hashes.")
if manifest_04a.get("input_03b_manifest_sha256") != sha256_file(INPUT_03B_MANIFEST):
    raise RuntimeError("04A was not built from the current 03B manifest.")
if manifest_04a.get("input_03c_manifest_sha256") != sha256_file(INPUT_03C["audit_manifest"]):
    raise RuntimeError("04A was not built from the current 03C manifest.")
if manifest_04a.get("selection_authority") != "03B_selected_models":
    raise RuntimeError("04A does not preserve 03B selection authority.")
if bool(manifest_04a.get("selection_mutated", True)):
    raise RuntimeError("04A reports a forbidden selection mutation.")
if bool(manifest_04a.get("model_refit", True)):
    raise RuntimeError("04A unexpectedly reports a refit.")
if bool(manifest_04a.get("threshold_resuggestion", True)):
    raise RuntimeError("04A unexpectedly reports threshold resuggestion.")
if bool(manifest_04a.get("weighted_score_used", True)):
    raise RuntimeError("04A unexpectedly reports a weighted score.")

model_reference_df = load_parquet(INPUT_04A["model_reference"])
if sha256_file(INPUT_04A["model_reference"]) != str(
    manifest_04a["output_artifacts"]["model_reference"]["sha256"]
):
    raise RuntimeError("04A model-reference hash mismatch.")

manifest_04b = None
if INPUT_04B_AUDIT.is_file():
    manifest_04b = json.loads(INPUT_04B_AUDIT.read_text(encoding="utf-8"))
    if str(manifest_04b.get("protocol_hash")) != PROTOCOL_HASH:
        raise RuntimeError("04B and the frozen protocol have different hashes.")
    if manifest_04b.get("input_03b_manifest_sha256") != sha256_file(INPUT_03B_MANIFEST):
        raise RuntimeError("04B was not built from the current 03B manifest.")
    if manifest_04b.get("input_04a_manifest_sha256") != sha256_file(INPUT_04A["audit_manifest"]):
        raise RuntimeError("04B was not built from the current 04A manifest.")
    if manifest_04b.get("selection_authority") != "03B_selected_models":
        raise RuntimeError("04B does not preserve 03B selection authority.")
    if bool(manifest_04b.get("selection_mutated", True)):
        raise RuntimeError("04B reports a forbidden selection mutation.")
    if bool(manifest_04b.get("model_refit", True)):
        raise RuntimeError("04B unexpectedly reports a model refit.")
    if bool(manifest_04b.get("threshold_resuggestion", True)):
        raise RuntimeError("04B unexpectedly reports threshold resuggestion.")
    if manifest_04b.get("downstream_selection_use") != "forbidden":
        raise RuntimeError("04B is not explicitly audit-only downstream.")
    print("04B audit manifest verified; no 04B scientific result table loaded.")
else:
    print("04B audit manifest absent; this does not alter the scientific 05 population.")

validation_protocol = json.loads(INPUT_04C["protocol"].read_text(encoding="utf-8"))
if str(validation_protocol.get("protocol_hash")) != PROTOCOL_HASH:
    raise RuntimeError("04C and the frozen protocol have different hashes.")
for key, expected in {
    "selection_mutated": False,
    "batch4_loaded": False,
    "threshold_resuggestion": False,
    "weighted_score_used": False,
    "cross_track_selection_or_ranking": False,
}.items():
    if validation_protocol.get(key) is not expected:
        raise RuntimeError(f"04C contract violation for {key}: {validation_protocol.get(key)!r}.")
if validation_protocol.get("natural_execution_key") != ["model_id", "random_state"]:
    raise RuntimeError("04C execution key is not canonical.")
if validation_protocol.get("decision_policy_key") != ["model_id", "random_state", "decision_scope"]:
    raise RuntimeError("04C decision-policy key is not canonical.")

for key in ("object_predictions", "pixel_predictions", "metrics", "spatial_component_metrics", "guardrails", "technical_events"):
    expected = str(validation_protocol["output_sha256"][key])
    observed = sha256_file(INPUT_04C[key])
    if observed != expected:
        raise RuntimeError(f"04C output hash mismatch for {key}.")

if set(track_contracts_df["track_id"].astype(str)) != expected_tracks:
    raise RuntimeError("03B track contracts must cover exactly E1-E8.")
if set(projection_eligibility_df["track_id"].astype(str)) != expected_tracks:
    raise RuntimeError("03C eligibility must cover exactly E1-E8.")

print("Upstream lineage verified through 04C.")


04B audit manifest verified; no 04B scientific result table loaded.
Upstream lineage verified through 04C.


## D — Reconstruction exacte de la frontière scientifique 04C → 05

In [4]:
validation_executions_df, selected_thresholds_df = build_validation_execution_registry(
    model_catalog=model_catalog_df,
    selected_models=selected_models_df,
    selected_runs=selected_runs_df,
    selected_thresholds=selected_threshold_rows_df,
    projection_eligibility=projection_eligibility_df,
    model_reference=model_reference_df,
    track_contracts=track_contracts_df,
)

execution_hash = sha256_dataframe(
    validation_executions_df.reindex(columns=expcfg.SIMCA_VALIDATION_EXECUTION_COLUMNS)
)
threshold_hash = sha256_dataframe(
    selected_thresholds_df.reindex(columns=expcfg.INTERNAL_CALIBRATION_SELECTED_THRESHOLD_COLUMNS)
)
if execution_hash != str(validation_protocol["execution_registry_sha256"]):
    raise RuntimeError("Reconstructed 04C execution registry hash mismatch.")
if threshold_hash != str(validation_protocol["selected_thresholds_sha256"]):
    raise RuntimeError("Reconstructed 04C threshold registry hash mismatch.")

validation_object_predictions_df = load_parquet(INPUT_04C["object_predictions"])
validation_pixel_predictions_df = load_parquet(INPUT_04C["pixel_predictions"])
validation_metrics_df = load_parquet(INPUT_04C["metrics"])
spatial_component_metrics_df = load_parquet(INPUT_04C["spatial_component_metrics"])
validation_guardrails_df = load_parquet(INPUT_04C["guardrails"])
validation_technical_events_df = load_parquet(INPUT_04C["technical_events"])

validated_04c = validate_robustness_inputs(
    validation_metrics_df,
    validation_guardrails_df,
    validation_executions_df,
    model_catalog_df,
    spatial_component_metrics_df,
    require_complete_tracks=True,
)

print("04C models:", validation_executions_df["model_id"].nunique())
print("04C executions:", len(validation_executions_df))
print("04C validation metric rows:", len(validation_metrics_df))


04C models: 38
04C executions: 78
04C validation metric rows: 7072


## E — Unités de modèle et Pareto de validation sur le panel 04C commun

Cette étape est la **seule** étape Pareto de 05. Les seeds 04C sont agrégées à poids égal au niveau du `model_id`. Le front est construit séparément dans chaque `track_id`. Les seeds supplémentaires n’existent pas encore à ce stade et ne pourront donc pas modifier ce front.


In [5]:
selection_units_df, selection_members_df = build_selection_unit_metrics(
    validation_metrics_df,
    validation_guardrails_df,
    validation_executions_df,
    model_catalog_df,
    spatial_component_metrics_df,
)

pareto_candidates_df, pareto_audit_df = build_pareto_diagnostics(
    selection_units_df,
    epsilon=float(expcfg.SIMCA_ROBUSTNESS_PARETO_EPSILON),
)

if selection_units_df.duplicated(["model_id", "track_id"]).any():
    raise RuntimeError("Selection units duplicate the scientific model key.")
if set(selection_units_df["track_id"].astype(str)) != expected_tracks:
    raise RuntimeError("Selection units do not cover exactly E1-E8.")
if set(pareto_audit_df["track_id"].astype(str)) != expected_tracks:
    raise RuntimeError("Pareto audit does not cover exactly E1-E8.")

for frame_name, frame in {
    "selection_units": selection_units_df,
    "selection_members": selection_members_df,
    "pareto_candidates": pareto_candidates_df,
    "pareto_audit": pareto_audit_df,
}.items():
    forbidden = set(expcfg.ACTIVE_PROTOCOL_FORBIDDEN_SCORE_COLUMNS).intersection(frame.columns)
    if forbidden:
        raise RuntimeError(f"Forbidden score columns in {frame_name}: {sorted(forbidden)}")

protocol_pareto_mask = pareto_candidates_df["is_protocol_pareto"].fillna(False).astype(bool)
PARETO_MODEL_IDS = tuple(
    sorted(pareto_candidates_df.loc[protocol_pareto_mask, "model_id"].astype(str).unique())
)
if not PARETO_MODEL_IDS:
    raise RuntimeError("No protocol Pareto model remains after base 04C validation.")

pareto_counts = (
    pareto_candidates_df.groupby("track_id", as_index=False)
    .agg(
        n_models=("model_id", "nunique"),
        n_protocol_pareto=("is_protocol_pareto", "sum"),
        n_diagnostic_pareto=("is_diagnostic_pareto", "sum"),
    )
    .sort_values("track_id")
)
display(pareto_counts)


,track_id,n_models,n_protocol_pareto,n_diagnostic_pareto
0,E1,2,0,1
1,E2,8,0,1
2,E3,1,0,1
3,E4,1,0,1
4,E5,1,1,1
5,E6,17,3,3
6,E7,1,0,1
7,E8,7,4,4


## F — Construction du panel de seeds supplémentaires

In [6]:
if bool(expcfg.SIMCA_ROBUSTNESS_RUN_ADDITIONAL_SEEDS):
    additional_seed_executions_df = build_robustness_seed_execution_registry(
        model_catalog_df,
        pareto_candidates_df,
        existing_executions=validation_executions_df,
        random_states=expcfg.SIMCA_ROBUSTNESS_ADDITIONAL_RANDOM_STATES,
    )
else:
    additional_seed_executions_df = pd.DataFrame(
        columns=expcfg.INTERNAL_CALIBRATION_EXECUTION_COLUMNS
    )

if len(additional_seed_executions_df):
    if additional_seed_executions_df.duplicated(["model_id", "random_state"]).any():
        raise RuntimeError("Additional seed executions duplicate their natural key.")
    if not stochastic_model_mask(additional_seed_executions_df).all():
        raise RuntimeError("A deterministic model entered the additional-seed panel.")
    if not set(additional_seed_executions_df["model_id"].astype(str)).issubset(PARETO_MODEL_IDS):
        raise RuntimeError("A non-Pareto model entered the additional-seed stress test.")
    if set(pd.to_numeric(additional_seed_executions_df["random_state"], errors="raise").astype(int)) != set(
        map(int, expcfg.SIMCA_ROBUSTNESS_ADDITIONAL_RANDOM_STATES)
    ):
        raise RuntimeError("The additional-seed panel differs from the frozen 05 contract.")

print("Protocol-Pareto models:", len(PARETO_MODEL_IDS))
print("Additional stochastic executions:", len(additional_seed_executions_df))


Protocol-Pareto models: 8
Additional stochastic executions: 56


## G — OOF batches 1–2 pour les nouvelles seeds et rematérialisation des seuils

Aucune politique n’est re-sélectionnée. Le moteur 03B est réutilisé uniquement pour produire les distributions OOF propres aux nouvelles seeds. Les coordonnées de politique choisies en 03B sont ensuite appliquées **exactement**.


In [7]:
allowed_batches = tuple(
    sorted(
        set(
            tuple(map(int, expcfg.SIMCA_CONCAT_REFIT_TRAIN_BATCHES))
            + tuple(map(int, expcfg.SIMCA_CONCAT_REFIT_PROJECTION_BATCHES))
        )
    )
)
forbidden_batches = set(map(int, expcfg.SIMCA_CONCAT_REFIT_FORBIDDEN_BATCHES))
if set(allowed_batches).intersection(forbidden_batches):
    raise RuntimeError("A forbidden batch entered the notebook-05 load contract.")
if not DB_H5_PATH.is_file():
    raise FileNotFoundError(DB_H5_PATH)

DATABASE_SHA256 = sha256_file(DB_H5_PATH)
object_db, image_db = load_nir_uco_h5(
    DB_H5_PATH,
    reconstruct_heavy_object_arrays=expcfg.SIMCA_CONCAT_REFIT_RECONSTRUCT_HEAVY_OBJECT_ARRAYS,
    batches=allowed_batches,
)
loaded_batches = {
    int(record["batch"])
    for record in object_db.values()
    if record.get("batch") is not None
}
if loaded_batches != set(allowed_batches):
    raise RuntimeError(
        f"Notebook 05 must load exactly batches {sorted(allowed_batches)}, got {sorted(loaded_batches)}."
    )
if loaded_batches.intersection(forbidden_batches):
    raise RuntimeError("A forbidden/test batch was loaded in notebook 05.")

if USE_WAVELENGTH_WINDOW:
    object_db, image_db, wavelengths, _ = select_wavelength_range_from_database(
        object_db=object_db,
        image_db=image_db,
        min_nm=expcfg.WAVELENGTH_WINDOW_MIN_NM,
        max_nm=expcfg.WAVELENGTH_WINDOW_MAX_NM,
    )
else:
    wavelengths = np.asarray(next(iter(object_db.values()))["wavelengths"], dtype=float)

additional_calibration_outputs = None
additional_threshold_metrics_df = pd.DataFrame(
    columns=expcfg.INTERNAL_CALIBRATION_THRESHOLD_METRIC_COLUMNS
)
additional_seed_thresholds_df = pd.DataFrame(
    columns=expcfg.SIMCA_ROBUSTNESS_SEED_THRESHOLD_COLUMNS
)

if len(additional_seed_executions_df):
    additional_configuration_hash = sha256_dataframe(
        additional_seed_executions_df.reindex(
            columns=expcfg.INTERNAL_CALIBRATION_EXECUTION_COLUMNS
        )
    )
    seed_calibration_context = {
        "protocol_hash": PROTOCOL_HASH,
        "pca_selection_fingerprint": str(manifest_03b["pca_selection_fingerprint"]),
        "track_contract_hash": str(manifest_03b["track_contract_hash"]),
        "fold_contract_hash": str(manifest_03b["fold_contract_hash"]),
        "configuration_hash": additional_configuration_hash,
    }

    additional_calibration_outputs = run_internal_calibration_8tracks(
        object_db=object_db,
        folds=folds_df,
        configurations=additional_seed_executions_df,
        wavelengths=wavelengths,
        target_class=expcfg.TARGET_CLASS,
        non_target_label=expcfg.NON_TARGET_LABEL,
        under_m_policy=expcfg.INTERNAL_CALIBRATION_UNDER_M_POLICY,
        verbose=expcfg.INTERNAL_CALIBRATION_VERBOSE,
        checkpoint_dir=SEED_CALIBRATION_CHECKPOINT_DIR,
        checkpoint_context=seed_calibration_context,
        resume_from_checkpoint=expcfg.INTERNAL_CALIBRATION_RESUME_FROM_CHECKPOINT,
        keep_oof_in_memory=False,
        keep_threshold_metrics_in_memory=True,
    )
    additional_threshold_metrics_df = additional_calibration_outputs["threshold_metrics"]
    if additional_threshold_metrics_df.empty:
        raise RuntimeError("Additional-seed OOF calibration returned no threshold metric.")

    additional_seed_thresholds_df = materialize_fixed_threshold_policy_for_runs(
        additional_threshold_metrics_df,
        selected_thresholds_df,
        model_catalog_df,
        expected_random_states=expcfg.SIMCA_ROBUSTNESS_ADDITIONAL_RANDOM_STATES,
        expected_model_ids=sorted(additional_seed_executions_df["model_id"].astype(str).unique()),
        expected_source_random_states=expcfg.SIMCA_ROBUSTNESS_BASE_RANDOM_STATES,
    )

    if additional_seed_thresholds_df.duplicated(
        ["model_id", "random_state", "decision_scope"]
    ).any():
        raise RuntimeError("Additional thresholds duplicate their natural key.")

print("Loaded batches:", sorted(loaded_batches))
print("Additional calibrated threshold rows:", len(additional_seed_thresholds_df))


Loaded batches: [1, 2, 3]
Additional calibrated threshold rows: 84


## H — Refit batches 1–2, projection batch 3 et évaluation des nouvelles seeds

In [8]:
additional_validation_executions_df = pd.DataFrame(
    columns=expcfg.SIMCA_VALIDATION_EXECUTION_COLUMNS
)
additional_object_predictions_df = pd.DataFrame(
    columns=expcfg.SIMCA_VALIDATION_OBJECT_PREDICTION_COLUMNS
)
additional_pixel_predictions_df = pd.DataFrame(
    columns=expcfg.SIMCA_VALIDATION_PIXEL_PREDICTION_COLUMNS
)
additional_technical_events_df = pd.DataFrame(
    columns=expcfg.SIMCA_VALIDATION_TECHNICAL_EVENT_COLUMNS
)
additional_validation_metrics_df = pd.DataFrame(
    columns=expcfg.SIMCA_VALIDATION_METRIC_COLUMNS
)
additional_spatial_component_metrics_df = pd.DataFrame(
    columns=expcfg.SIMCA_SPATIAL_COMPONENT_METRIC_COLUMNS
)
additional_validation_guardrails_df = pd.DataFrame(
    columns=expcfg.SIMCA_VALIDATION_GUARDRAIL_COLUMNS
)
additional_seed_metrics_df = pd.DataFrame(
    columns=expcfg.SIMCA_ROBUSTNESS_SELECTION_MEMBER_COLUMNS
)
ADDITIONAL_VALIDATION_PLAN_HASH = ""
ADDITIONAL_EXECUTION_REGISTRY_HASH = ""
ADDITIONAL_VALIDATION_CHECKPOINT_RUN_DIR = None

if len(additional_seed_executions_df):
    model_status = (
        validation_executions_df[
            ["model_id", "eligibility_status", "downstream_status"]
        ]
        .drop_duplicates()
    )
    if model_status["model_id"].astype(str).duplicated().any():
        raise RuntimeError("04C model eligibility/downstream status is not invariant.")

    additional_validation_executions_df = (
        additional_seed_executions_df.merge(
            model_status,
            on="model_id",
            how="left",
            validate="many_to_one",
        )
        .reindex(columns=expcfg.SIMCA_VALIDATION_EXECUTION_COLUMNS)
        .sort_values(["track_id", "model_id", "random_state"], kind="mergesort")
        .reset_index(drop=True)
    )
    if additional_validation_executions_df[
        ["eligibility_status", "downstream_status"]
    ].isna().any().any():
        raise RuntimeError("Additional executions lost their frozen 03C/04A status.")

    ADDITIONAL_EXECUTION_REGISTRY_HASH = sha256_dataframe(
        additional_validation_executions_df
    )
    ADDITIONAL_VALIDATION_PLAN_HASH = sha256_payload(
        {
            "role": "notebook05_additional_seed_batch3_stress_test",
            "global_protocol_hash": PROTOCOL_HASH,
            "child_contract_hash": ROBUSTNESS_CONTRACT_HASH,
            "04c_validation_evaluation_rule_hash": str(
                validation_protocol["validation_evaluation_rule_hash"]
            ),
            "train_batches": list(map(int, expcfg.SIMCA_CONCAT_REFIT_TRAIN_BATCHES)),
            "projection_batches": list(map(int, expcfg.SIMCA_CONCAT_REFIT_PROJECTION_BATCHES)),
            "model_ids": sorted(additional_validation_executions_df["model_id"].astype(str).unique()),
            "random_states": list(map(int, expcfg.SIMCA_ROBUSTNESS_ADDITIONAL_RANDOM_STATES)),
        }
    )
    checkpoint_context = {
        "protocol_hash": PROTOCOL_HASH,
        "validation_plan_hash": ADDITIONAL_VALIDATION_PLAN_HASH,
        "execution_registry_hash": ADDITIONAL_EXECUTION_REGISTRY_HASH,
        "database_sha256": DATABASE_SHA256,
    }
    refit_kwargs = {
        "wavelengths": wavelengths,
        "train_batches": expcfg.SIMCA_CONCAT_REFIT_TRAIN_BATCHES,
        "projection_batches": expcfg.SIMCA_CONCAT_REFIT_PROJECTION_BATCHES,
        "target_class": expcfg.TARGET_CLASS,
        "non_target_label": expcfg.NON_TARGET_LABEL,
        "border_width": expcfg.SIMCA_CONCAT_REFIT_BORDER_WIDTH,
        "verbose": expcfg.SIMCA_CONCAT_REFIT_VERBOSE,
    }

    if expcfg.SIMCA_CONCAT_REFIT_CHECKPOINT_ENABLED:
        additional_refit_outputs = run_locked_simca_validation_refit_checkpointed(
            additional_validation_executions_df,
            object_db=object_db,
            checkpoint_dir=SEED_VALIDATION_CHECKPOINT_DIR,
            checkpoint_context=checkpoint_context,
            resume=expcfg.SIMCA_CONCAT_REFIT_RESUME_FROM_CHECKPOINT,
            **refit_kwargs,
        )
        ADDITIONAL_VALIDATION_CHECKPOINT_RUN_DIR = Path(
            additional_refit_outputs["checkpoint_run_dir"]
        )
    else:
        additional_refit_outputs = run_locked_simca_validation_refit(
            additional_validation_executions_df,
            object_db=object_db,
            **refit_kwargs,
        )

    additional_object_predictions_df = additional_refit_outputs["object_predictions"]
    additional_pixel_predictions_df = additional_refit_outputs["pixel_predictions"]
    additional_technical_events_df = additional_refit_outputs["technical_events"]

    additional_validation_metrics_df = evaluate_locked_validation_predictions(
        additional_validation_executions_df,
        additional_seed_thresholds_df,
        additional_object_predictions_df,
        additional_pixel_predictions_df,
        technical_events=additional_technical_events_df,
    )

    # The 04C spatial helper validates the complete set of locked supported
    # pixel tracks. Additional seeds are Pareto-only and may cover only a
    # subset. Add one frozen 04C anchor execution for each missing locked track
    # solely for that integrity check, then discard every anchor output.
    locked_track_ids = set(map(str, spatial_lock.get("spatial_track_ids", ())))
    additional_supported_pixel = additional_validation_executions_df.loc[
        additional_validation_executions_df["projection_level"].astype(str).eq("pixel_projection")
        & additional_validation_executions_df["downstream_status"].astype(str).eq("supported")
        & additional_validation_executions_df["eligibility_status"].astype(str).isin(
            expcfg.SIMCA_CONCAT_REFIT_SUPPORTED_ELIGIBILITY_STATUSES
        )
    ]
    represented_locked_tracks = set(additional_supported_pixel["track_id"].astype(str))
    missing_locked_tracks = locked_track_ids - represented_locked_tracks

    spatial_anchor_executions_df = (
        validation_executions_df.loc[
            validation_executions_df["track_id"].astype(str).isin(missing_locked_tracks)
            & validation_executions_df["projection_level"].astype(str).eq("pixel_projection")
            & validation_executions_df["downstream_status"].astype(str).eq("supported")
            & validation_executions_df["eligibility_status"].astype(str).isin(
                expcfg.SIMCA_CONCAT_REFIT_SUPPORTED_ELIGIBILITY_STATUSES
            )
        ]
        .sort_values(["track_id", "model_id", "random_state"], kind="mergesort")
        .groupby("track_id", as_index=False, sort=False)
        .head(1)
    )
    if set(spatial_anchor_executions_df["track_id"].astype(str)) != missing_locked_tracks:
        raise RuntimeError(
            "Could not construct frozen 04C spatial anchors for every missing locked track."
        )

    spatial_context_executions_df = pd.concat(
        [additional_validation_executions_df, spatial_anchor_executions_df],
        ignore_index=True,
        sort=False,
    ).reindex(columns=expcfg.SIMCA_VALIDATION_EXECUTION_COLUMNS)
    anchor_keys = spatial_anchor_executions_df[["model_id", "random_state"]].drop_duplicates()
    anchor_thresholds_df = selected_thresholds_df.merge(
        anchor_keys,
        on=["model_id", "random_state"],
        how="inner",
        validate="many_to_one",
    )
    spatial_context_thresholds_df = pd.concat(
        [additional_seed_thresholds_df, anchor_thresholds_df],
        ignore_index=True,
        sort=False,
    ).reindex(columns=expcfg.INTERNAL_CALIBRATION_SELECTED_THRESHOLD_COLUMNS)
    anchor_projection_ids = set(spatial_anchor_executions_df["projection_id"].astype(str))
    anchor_pixel_predictions_df = validation_pixel_predictions_df.loc[
        validation_pixel_predictions_df["projection_id"].astype(str).isin(anchor_projection_ids)
    ]
    spatial_context_pixel_predictions_df = pd.concat(
        [additional_pixel_predictions_df, anchor_pixel_predictions_df],
        ignore_index=True,
        sort=False,
    ).reindex(columns=expcfg.SIMCA_VALIDATION_PIXEL_PREDICTION_COLUMNS)

    additional_spatial_outputs = build_locked_spatial_validation_outputs(
        spatial_context_executions_df,
        spatial_context_thresholds_df,
        spatial_context_pixel_predictions_df,
        image_db,
        spatial_lock,
    )
    additional_run_keys = additional_validation_executions_df[
        ["model_id", "random_state"]
    ].drop_duplicates()
    additional_spatial_component_metrics_df = additional_spatial_outputs[
        "spatial_component_metrics"
    ].merge(
        additional_run_keys,
        on=["model_id", "random_state"],
        how="inner",
        validate="many_to_one",
    ).reindex(columns=expcfg.SIMCA_SPATIAL_COMPONENT_METRIC_COLUMNS)

    additional_validation_guardrails_df = build_validation_guardrails(
        additional_validation_executions_df,
        additional_validation_metrics_df,
        spatial_component_metrics=additional_spatial_component_metrics_df,
    )

    additional_seed_metrics_df = build_seed_metrics(
        additional_validation_metrics_df,
        additional_validation_guardrails_df,
        additional_validation_executions_df,
        model_catalog_df,
        additional_spatial_component_metrics_df,
    )

print("Additional batch-3 metric rows:", len(additional_validation_metrics_df))
print("Additional robustness member rows:", len(additional_seed_metrics_df))


Additional batch-3 metric rows: 5712
Additional robustness member rows: 84


## I — Panel complet de robustesse, désaccord entre seeds et stabilité

Le Pareto reste celui de la section E. Ici, on assemble uniquement le panel de stress-test des **modèles déjà Pareto**. Les modèles stochastiques reçoivent le panel étendu de seeds ; les modèles déterministes conservent uniquement leur exécution canonique 03B/04C, ne sont jamais clonés et ne sont pas soumis au résumé de variabilité inter-seeds (`not_applicable_deterministic` est porté par la revue).

La stabilité bloquante est limitée aux métriques de risque primaire déclarées par track et au désaccord de décision configuré. Les autres instabilités restent des warnings.


In [14]:
pareto_ids = set(PARETO_MODEL_IDS)

base_pareto_executions_df = validation_executions_df.loc[
    validation_executions_df["model_id"].astype(str).isin(pareto_ids)
].copy()
base_pareto_thresholds_df = selected_thresholds_df.loc[
    selected_thresholds_df["model_id"].astype(str).isin(pareto_ids)
].copy()
base_pareto_seed_metrics_df = selection_members_df.loc[
    selection_members_df["model_id"].astype(str).isin(pareto_ids)
].copy()

robustness_validation_executions_df = pd.concat(
    [base_pareto_executions_df, additional_validation_executions_df],
    ignore_index=True,
    sort=False,
).reindex(columns=expcfg.SIMCA_VALIDATION_EXECUTION_COLUMNS)
if robustness_validation_executions_df.duplicated(["model_id", "random_state"]).any():
    raise RuntimeError("Complete robustness execution panel duplicates a natural key.")

robustness_seed_executions_df = (
    robustness_validation_executions_df
    .reindex(columns=expcfg.SIMCA_ROBUSTNESS_SEED_EXECUTION_COLUMNS)
    .sort_values(["model_id", "random_state"], kind="mergesort")
    .reset_index(drop=True)
)

robustness_seed_thresholds_df = pd.concat(
    [base_pareto_thresholds_df, additional_seed_thresholds_df],
    ignore_index=True,
    sort=False,
).reindex(columns=expcfg.SIMCA_ROBUSTNESS_SEED_THRESHOLD_COLUMNS)
robustness_seed_thresholds_df = (
    robustness_seed_thresholds_df
    .sort_values(
        ["model_id", "random_state", "decision_scope"],
        kind="mergesort",
    )
    .reset_index(drop=True)
)
if robustness_seed_thresholds_df.duplicated(
    ["model_id", "random_state", "decision_scope"]
).any():
    raise RuntimeError("Complete seed thresholds duplicate their natural key.")

robustness_seed_metrics_df = pd.concat(
    [base_pareto_seed_metrics_df, additional_seed_metrics_df],
    ignore_index=True,
    sort=False,
).reindex(columns=expcfg.SIMCA_ROBUSTNESS_SELECTION_MEMBER_COLUMNS)
if robustness_seed_metrics_df.duplicated(
    ["model_id", "random_state", "track_id", "decision_scope"]
).any():
    raise RuntimeError("Complete seed metrics duplicate their execution-scope key.")

base_projection_ids = set(base_pareto_executions_df["projection_id"].astype(str))
base_object_predictions_df = validation_object_predictions_df.loc[
    validation_object_predictions_df["projection_id"].astype(str).isin(
        base_projection_ids
    )
].copy()
base_pixel_predictions_df = validation_pixel_predictions_df.loc[
    validation_pixel_predictions_df["projection_id"].astype(str).isin(
        base_projection_ids
    )
].copy()

robustness_object_predictions_df = pd.concat(
    [base_object_predictions_df, additional_object_predictions_df],
    ignore_index=True,
    sort=False,
).reindex(columns=expcfg.SIMCA_VALIDATION_OBJECT_PREDICTION_COLUMNS)
robustness_pixel_predictions_df = pd.concat(
    [base_pixel_predictions_df, additional_pixel_predictions_df],
    ignore_index=True,
    sort=False,
).reindex(columns=expcfg.SIMCA_VALIDATION_PIXEL_PREDICTION_COLUMNS)

seed_disagreement_df = compute_seed_decision_disagreement(
    robustness_validation_executions_df,
    robustness_seed_thresholds_df,
    robustness_object_predictions_df,
    robustness_pixel_predictions_df,
)

stochastic_seed_metrics_df = robustness_seed_metrics_df.loc[
    robustness_seed_metrics_df["is_stochastic"].fillna(False).astype(bool)
].copy()

stability_summary_df = summarize_random_state_stability_metrics(
    stochastic_seed_metrics_df,
    decision_disagreement=seed_disagreement_df,
    expected_random_states=expcfg.SIMCA_ROBUSTNESS_RANDOM_STATES,
)

threshold_stability_df = build_threshold_stability_diagnostics(
    robustness_seed_thresholds_df,
    robustness_validation_executions_df,
    raise_on_policy_drift=False,
)

# Non-regression checks requested by the 05 contract.
pareto_catalog = model_catalog_df.loc[
    model_catalog_df["model_id"].astype(str).isin(pareto_ids)
].copy()
pareto_catalog["is_stochastic"] = stochastic_model_mask(pareto_catalog)
stochastic_ids = set(
    pareto_catalog.loc[
        pareto_catalog["is_stochastic"],
        "model_id",
    ].astype(str)
)
deterministic_ids = pareto_ids - stochastic_ids

seed_sets = (
    robustness_seed_executions_df
    .groupby("model_id")["random_state"]
    .agg(lambda values: set(map(int, values)))
)
expected_stochastic_states = set(
    map(int, expcfg.SIMCA_ROBUSTNESS_RANDOM_STATES)
)
if stochastic_ids:
    incomplete = {
        model_id: sorted(seed_sets.get(model_id, set()))
        for model_id in sorted(stochastic_ids)
        if seed_sets.get(model_id, set()) != expected_stochastic_states
    }
    if incomplete:
        raise RuntimeError(
            "A stochastic Pareto model does not have the complete "
            f"robustness seed panel: {incomplete}."
        )

additional_states = set(
    map(int, expcfg.SIMCA_ROBUSTNESS_ADDITIONAL_RANDOM_STATES)
)
if deterministic_ids:
    cloned = {
        model_id: sorted(seed_sets.get(model_id, set()).intersection(additional_states))
        for model_id in sorted(deterministic_ids)
        if seed_sets.get(model_id, set()).intersection(additional_states)
    }
    if cloned:
        raise RuntimeError(
            "A deterministic Pareto model was unnecessarily cloned onto "
            f"additional robustness seeds: {cloned}."
        )

if len(additional_seed_executions_df):
    probe = additional_seed_executions_df.iloc[0]
    base = validation_executions_df.loc[
        validation_executions_df["model_id"].astype(str).eq(
            str(probe["model_id"])
        )
        & validation_executions_df["random_state"].eq(
            int(expcfg.SIMCA_ROBUSTNESS_BASE_RANDOM_STATES[0])
        )
    ]
    if len(base) != 1:
        raise RuntimeError(
            "Seed identity non-regression probe could not resolve "
            "its base execution."
        )
    base = base.iloc[0]
    if str(base["fit_id"]) == str(probe["fit_id"]):
        raise RuntimeError(
            "A new stochastic seed unexpectedly reused the base fit_id."
        )
    if str(base["projection_id"]) == str(probe["projection_id"]):
        raise RuntimeError(
            "A new stochastic seed unexpectedly reused the base projection_id."
        )

if len(threshold_stability_df) and not threshold_stability_df[
    "policy_coordinates_invariant"
].fillna(False).astype(bool).all():
    raise RuntimeError(
        "A frozen 03B threshold-policy coordinate changed in notebook 05."
    )

print("Robustness executions:", len(robustness_seed_executions_df))
print("Stochastic Pareto models:", len(stochastic_ids))
print("Deterministic Pareto models:", len(deterministic_ids))
display(
    stability_summary_df[
        [
            column
            for column in (
                "model_id",
                "track_id",
                "metric",
                "stability_role",
                "n_random_states",
                "mean",
                "std",
                "min",
                "max",
                "blocking_metric_failure",
                "supporting_metric_warning",
                "model_stability_status",
                "stability_flags",
            )
            if column in stability_summary_df.columns
        ]
    ].head(60)
)
display(
    threshold_stability_df[
        [
            column
            for column in (
                "model_id",
                "track_id",
                "decision_scope",
                "decision_mode",
                "n_random_states",
                "policy_coordinates_invariant",
                "fixed_numeric_threshold_invariant",
                "threshold_center_range",
                "uncertainty_band_width_mean",
                "band_width_cv",
                "center_range_over_mean_width",
                "numeric_stability_warning",
                "stability_status",
            )
            if column in threshold_stability_df.columns
        ]
    ].head(60)
)


Robustness executions: 80
Stochastic Pareto models: 8
Deterministic Pareto models: 0


,model_id,track_id,metric,stability_role,n_random_states,mean,std,min,max,blocking_metric_failure,supporting_metric_warning,model_stability_status,stability_flags
0,model_0326bb57ce83ca64ed15,E6,direct__target_miss_rate,blocking_primary_risk,10,0.000000,0.000000,0.000000,0.000000,False,False,robust_with_supporting_warnings,decision_disagreement_rate;direct__macro_image...
1,model_1b2c23b63e3e3a4c975a,E5,direct__target_miss_rate,blocking_primary_risk,10,0.013208,0.015533,0.000000,0.037736,False,False,robust,
2,model_23963a68a13219879027,E6,direct__target_miss_rate,blocking_primary_risk,10,0.000000,0.000000,0.000000,0.000000,False,False,robust_with_supporting_warnings,decision_disagreement_rate;direct__macro_image...
3,model_3ba4782ef74eb4daaa5d,E8,direct__target_miss_rate,supporting_secondary,10,0.005700,0.001302,0.004071,0.008456,False,False,unstable_blocking,decision_disagreement_rate;pixel_to_object__de...
4,model_3ba4782ef74eb4daaa5d,E8,pixel_to_object__target_miss_rate,supporting_secondary,10,0.000000,0.000000,0.000000,0.000000,False,False,unstable_blocking,decision_disagreement_rate;pixel_to_object__de...
5,model_54a8a3135519ad96284c,E8,direct__target_miss_rate,supporting_secondary,10,0.008988,0.002591,0.005324,0.012527,False,False,unstable_blocking,decision_disagreement_rate;direct__decided_bal...
6,model_54a8a3135519ad96284c,E8,pixel_to_object__target_miss_rate,supporting_secondary,10,0.000000,0.000000,0.000000,0.000000,False,False,unstable_blocking,decision_disagreement_rate;direct__decided_bal...
7,model_8ce785efa5b5327a9667,E6,direct__target_miss_rate,blocking_primary_risk,10,0.000000,0.000000,0.000000,0.000000,False,False,unstable_blocking,target_decision_disagreement_rate
8,model_b8297b7d0756ad8306ae,E8,direct__target_miss_rate,supporting_secondary,10,0.009302,0.000533,0.008143,0.010022,False,False,robust_with_supporting_warnings,decision_disagreement_rate;pixel_to_object__de...
9,model_b8297b7d0756ad8306ae,E8,pixel_to_object__target_miss_rate,supporting_secondary,10,0.000000,0.000000,0.000000,0.000000,False,False,robust_with_supporting_warnings,decision_disagreement_rate;pixel_to_object__de...


,model_id,track_id,decision_scope,decision_mode,n_random_states,policy_coordinates_invariant,fixed_numeric_threshold_invariant,threshold_center_range,uncertainty_band_width_mean,band_width_cv,center_range_over_mean_width,numeric_stability_warning,stability_status
0,model_0326bb57ce83ca64ed15,E6,direct,3way,10,True,False,0.194568,0.492581,0.214430,0.394996,True,supporting_numeric_threshold_warning
1,model_1b2c23b63e3e3a4c975a,E5,direct,2way,10,True,True,0.000000,0.000000,NaN,NaN,False,stable
2,model_23963a68a13219879027,E6,direct,3way,10,True,False,0.210915,0.167895,0.738865,1.256228,True,supporting_numeric_threshold_warning
3,model_3ba4782ef74eb4daaa5d,E8,direct,3way,10,True,False,0.048624,0.779022,0.043340,0.062416,False,stable
4,model_3ba4782ef74eb4daaa5d,E8,pixel_to_object,3way,10,True,False,0.081979,0.422820,0.122685,0.193886,False,stable
5,model_54a8a3135519ad96284c,E8,direct,3way,10,True,False,0.063734,0.907335,0.049117,0.070243,False,stable
6,model_54a8a3135519ad96284c,E8,pixel_to_object,3way,10,True,False,0.140147,0.135897,0.771790,1.031276,True,supporting_numeric_threshold_warning
7,model_8ce785efa5b5327a9667,E6,direct,3way,10,True,False,0.186200,1.714570,0.079129,0.108598,False,stable
8,model_b8297b7d0756ad8306ae,E8,direct,3way,10,True,False,0.052189,1.030619,0.044278,0.050639,False,stable
9,model_b8297b7d0756ad8306ae,E8,pixel_to_object,3way,10,True,False,0.087558,0.109524,0.529078,0.799444,True,supporting_numeric_threshold_warning


In [ ]:
bad_models = {
    "model_3ba4782ef74eb4daaa5d",
    "model_54a8a3135519ad96284c",
    "model_8ce785efa5b5327a9667",
    "model_c8d347632e1452f6d9af",
}

debug_thresholds = (
    robustness_seed_thresholds_df.loc[
        robustness_seed_thresholds_df["model_id"]
        .astype(str)
        .isin(bad_models),
        [
            "model_id",
            "random_state",
            "decision_scope",
            "lower_quantile",
            "upper_quantile",
            "vote_threshold",
            "lower_threshold",
            "upper_threshold",
        ],
    ]
    .sort_values(
        ["model_id", "decision_scope", "random_state"],
        kind="mergesort",
    )
)

display(debug_thresholds)

for (model_id, scope), group in debug_thresholds.groupby(
    ["model_id", "decision_scope"],
    sort=False,
):
    print("\n", model_id, scope)

    for column in (
        "lower_quantile",
        "upper_quantile",
        "vote_threshold",
    ):
        values = pd.to_numeric(
            group[column],
            errors="coerce",
        )

        print(
            column,
            [
                None if pd.isna(value)
                else repr(float(value))
                for value in values
            ],
        )

,model_id,random_state,decision_scope,lower_quantile,upper_quantile,vote_threshold,lower_threshold,upper_threshold
30,model_3ba4782ef74eb4daaa5d,0,direct,0.9,0.25,NaN,-0.225718,0.553662
32,model_3ba4782ef74eb4daaa5d,1,direct,0.9,0.25,NaN,-0.216919,0.570035
34,model_3ba4782ef74eb4daaa5d,2,direct,0.9,0.25,NaN,-0.160832,0.546113
36,model_3ba4782ef74eb4daaa5d,3,direct,0.9,0.25,NaN,-0.261539,0.561242
38,model_3ba4782ef74eb4daaa5d,4,direct,0.9,0.25,NaN,-0.208088,0.551792
40,model_3ba4782ef74eb4daaa5d,5,direct,0.9,0.25,NaN,-0.232416,0.549388
42,model_3ba4782ef74eb4daaa5d,10,direct,0.9,0.25,NaN,-0.201238,0.564593
44,model_3ba4782ef74eb4daaa5d,20,direct,0.9,0.25,NaN,-0.240713,0.558713
46,model_3ba4782ef74eb4daaa5d,42,direct,0.9,0.25,NaN,-0.187526,0.576959
48,model_3ba4782ef74eb4daaa5d,100,direct,0.9,0.25,NaN,-0.265273,0.557459



 model_3ba4782ef74eb4daaa5d direct
lower_quantile ['0.8999999761581421', '0.8999999761581421', '0.8999999761581421', '0.9', '0.9', '0.9', '0.9', '0.9', '0.9', '0.9']
upper_quantile ['0.25', '0.25', '0.25', '0.25', '0.25', '0.25', '0.25', '0.25', '0.25', '0.25']
vote_threshold [None, None, None, None, None, None, None, None, None, None]

 model_3ba4782ef74eb4daaa5d pixel_to_object
lower_quantile ['1.0', '1.0', '1.0', '1.0', '1.0', '1.0', '1.0', '1.0', '1.0', '1.0']
upper_quantile ['0.10000000149011612', '0.10000000149011612', '0.10000000149011612', '0.1', '0.1', '0.1', '0.1', '0.1', '0.1', '0.1']
vote_threshold [None, None, None, None, None, None, None, None, None, None]

 model_54a8a3135519ad96284c direct
lower_quantile ['0.8999999761581421', '0.8999999761581421', '0.8999999761581421', '0.9', '0.9', '0.9', '0.9', '0.9', '0.9', '0.9']
upper_quantile ['0.25', '0.25', '0.25', '0.25', '0.25', '0.25', '0.25', '0.25', '0.25', '0.25']
vote_threshold [None, None, None, None, None, None, None,

## J — Diagnostics supporting de robustesse

Les analyses de cette section sont **descriptives/supporting**. Elles ne recalculent pas le Pareto officiel, ne changent pas les guardrails, ne modifient pas le verrou spatial 03C et ne participent pas à l’éligibilité pré-batch4.

Elles évaluent successivement :

1. la sensibilité locale aux seuils et l’influence d’une image source ;
2. la stabilité du front Pareto au retrait d’une seed de base ;
3. la sensibilité locale du verrou spatial 03C ;
4. la sensibilité au choix admissible des folds de calibration.


### J1 — Sensibilité aux seuils et à la composition du batch 3


In [15]:
combined_validation_metrics_df = pd.concat(
    [
        validation_metrics_df.loc[
            validation_metrics_df["model_id"].astype(str).isin(pareto_ids)
        ],
        additional_validation_metrics_df,
    ],
    ignore_index=True,
    sort=False,
).reindex(columns=expcfg.SIMCA_VALIDATION_METRIC_COLUMNS)

robustness_spatial_component_metrics_df = pd.concat(
    [
        spatial_component_metrics_df.loc[
            spatial_component_metrics_df["model_id"].astype(str).isin(
                pareto_ids
            )
        ],
        additional_spatial_component_metrics_df,
    ],
    ignore_index=True,
    sort=False,
).reindex(columns=expcfg.SIMCA_SPATIAL_COMPONENT_METRIC_COLUMNS)

threshold_sensitivity_plan_df = build_threshold_sensitivity_plan(
    robustness_validation_executions_df,
    robustness_seed_thresholds_df,
    model_ids=sorted(pareto_ids),
)

threshold_sensitivity_outputs = evaluate_threshold_sensitivity(
    threshold_sensitivity_plan_df,
    robustness_seed_thresholds_df,
    robustness_validation_executions_df,
    robustness_object_predictions_df,
    robustness_pixel_predictions_df,
    reference_validation_metrics=combined_validation_metrics_df,
)
threshold_sensitivity_metrics_df = threshold_sensitivity_outputs["metrics"]
threshold_sensitivity_decisions_df = threshold_sensitivity_outputs["decisions"]

source_image_influence_df = build_source_image_influence_diagnostics(
    combined_validation_metrics_df,
    model_ids=sorted(pareto_ids),
)

for name, frame in {
    "threshold_sensitivity_plan": threshold_sensitivity_plan_df,
    "threshold_sensitivity_metrics": threshold_sensitivity_metrics_df,
    "threshold_sensitivity_decisions": threshold_sensitivity_decisions_df,
    "threshold_stability": threshold_stability_df,
    "source_image_influence": source_image_influence_df,
}.items():
    assert_supporting_only(frame, name=name)

print("Threshold perturbations:", len(threshold_sensitivity_plan_df))
print("Threshold-sensitivity metric rows:", len(threshold_sensitivity_metrics_df))
print("Threshold-sensitivity decision rows:", len(threshold_sensitivity_decisions_df))
print("Leave-one-source-image-out rows:", len(source_image_influence_df))

if len(threshold_sensitivity_metrics_df):
    display(
        threshold_sensitivity_metrics_df.sort_values(
            ["track_id", "model_id", "random_state", "decision_scope", "metric"],
            kind="mergesort",
        ).head(60)
    )

if len(source_image_influence_df):
    display(
        source_image_influence_df.sort_values(
            ["absolute_delta"],
            ascending=False,
            kind="mergesort",
        ).head(40)
    )


Threshold perturbations: 460
Threshold-sensitivity metric rows: 7820
Threshold-sensitivity decision rows: 460
Leave-one-source-image-out rows: 2160


,model_id,random_state,track_id,decision_scope,perturbation_type,perturbation_value,metric,reference_value,alternative_value,delta,practical_tolerance,effect_status,selection_influence
4760,model_1b2c23b63e3e3a4c975a,0,E5,direct,direct_threshold_delta,-0.05,balanced_accuracy,0.936021,0.945455,9.433933e-03,0.03,within_practical_tolerance,False
4930,model_1b2c23b63e3e3a4c975a,0,E5,direct,direct_threshold_delta,0.05,balanced_accuracy,0.936021,0.963293,2.727270e-02,0.03,within_practical_tolerance,False
4761,model_1b2c23b63e3e3a4c975a,0,E5,direct,direct_threshold_delta,-0.05,coverage_rate,1.000000,1.000000,0.000000e+00,0.05,within_practical_tolerance,False
4931,model_1b2c23b63e3e3a4c975a,0,E5,direct,direct_threshold_delta,0.05,coverage_rate,1.000000,1.000000,0.000000e+00,0.05,within_practical_tolerance,False
4762,model_1b2c23b63e3e3a4c975a,0,E5,direct,direct_threshold_delta,-0.05,decided_balanced_accuracy,0.936021,0.945455,9.433933e-03,0.03,within_practical_tolerance,False
4932,model_1b2c23b63e3e3a4c975a,0,E5,direct,direct_threshold_delta,0.05,decided_balanced_accuracy,0.936021,0.963293,2.727270e-02,0.03,within_practical_tolerance,False
4763,model_1b2c23b63e3e3a4c975a,0,E5,direct,direct_threshold_delta,-0.05,false_accept_rate,0.109091,0.109091,-2.709302e-10,0.05,within_practical_tolerance,False
4933,model_1b2c23b63e3e3a4c975a,0,E5,direct,direct_threshold_delta,0.05,false_accept_rate,0.109091,0.054545,-5.454545e-02,0.05,outside_practical_tolerance,False
4764,model_1b2c23b63e3e3a4c975a,0,E5,direct,direct_threshold_delta,-0.05,macro_image_balanced_accuracy,0.936021,0.945455,9.433933e-03,0.03,within_practical_tolerance,False
4934,model_1b2c23b63e3e3a4c975a,0,E5,direct,direct_threshold_delta,0.05,macro_image_balanced_accuracy,0.936021,0.963293,2.727270e-02,0.03,within_practical_tolerance,False


,model_id,random_state,track_id,decision_scope,metric,omitted_source_image,n_source_images,n_finite_source_images_full,n_finite_source_images_retained,full_macro_image_value,leave_one_image_out_value,delta,absolute_delta,influence_status,selection_influence
0,model_0326bb57ce83ca64ed15,0,E6,direct,balanced_accuracy,almond3,2,0,0,NaN,NaN,NaN,NaN,not_estimable_class_coverage_lost,False
1,model_0326bb57ce83ca64ed15,0,E6,direct,balanced_accuracy,peanut3,2,0,0,NaN,NaN,NaN,NaN,not_estimable_class_coverage_lost,False
2,model_0326bb57ce83ca64ed15,0,E6,direct,coverage_rate,almond3,2,2,1,0.890909,NaN,NaN,NaN,not_estimable_class_coverage_lost,False
3,model_0326bb57ce83ca64ed15,0,E6,direct,coverage_rate,peanut3,2,2,1,0.890909,NaN,NaN,NaN,not_estimable_class_coverage_lost,False
4,model_0326bb57ce83ca64ed15,0,E6,direct,decided_balanced_accuracy,almond3,2,0,0,NaN,NaN,NaN,NaN,not_estimable_class_coverage_lost,False
5,model_0326bb57ce83ca64ed15,0,E6,direct,decided_balanced_accuracy,peanut3,2,0,0,NaN,NaN,NaN,NaN,not_estimable_class_coverage_lost,False
6,model_0326bb57ce83ca64ed15,0,E6,direct,false_accept_rate,almond3,2,1,0,0.181818,NaN,NaN,NaN,not_estimable_class_coverage_lost,False
7,model_0326bb57ce83ca64ed15,0,E6,direct,false_accept_rate,peanut3,2,1,1,0.181818,NaN,NaN,NaN,not_estimable_class_coverage_lost,False
8,model_0326bb57ce83ca64ed15,0,E6,direct,macro_object_target_miss_rate,almond3,2,1,1,0.000000,NaN,NaN,NaN,not_estimable_class_coverage_lost,False
9,model_0326bb57ce83ca64ed15,0,E6,direct,macro_object_target_miss_rate,peanut3,2,1,0,0.000000,NaN,NaN,NaN,not_estimable_class_coverage_lost,False


### J2 — Robustesse du front Pareto


In [16]:
if bool(expcfg.SIMCA_ROBUSTNESS_RUN_PARETO_FRONT_SENSITIVITY):
    (
        pareto_robustness_replicates_df,
        pareto_robustness_summary_df,
        pareto_robustness_audit_df,
    ) = build_pareto_front_robustness(
        selection_members_df,
        pareto_candidates_df,
        base_random_states=expcfg.SIMCA_ROBUSTNESS_BASE_RANDOM_STATES,
        epsilon=float(expcfg.SIMCA_ROBUSTNESS_PARETO_EPSILON),
    )
else:
    pareto_robustness_replicates_df = pd.DataFrame(
        columns=expcfg.SIMCA_ROBUSTNESS_PARETO_ROBUSTNESS_REPLICATE_COLUMNS
    )
    pareto_robustness_summary_df = pd.DataFrame(
        columns=expcfg.SIMCA_ROBUSTNESS_PARETO_ROBUSTNESS_SUMMARY_COLUMNS
    )
    pareto_robustness_audit_df = pd.DataFrame(
        columns=expcfg.SIMCA_ROBUSTNESS_PARETO_ROBUSTNESS_AUDIT_COLUMNS
    )

for name, frame in {
    "pareto_robustness_replicates": pareto_robustness_replicates_df,
    "pareto_robustness_summary": pareto_robustness_summary_df,
    "pareto_robustness_audit": pareto_robustness_audit_df,
}.items():
    assert_supporting_only(frame, name=name)

if len(pareto_robustness_summary_df):
    display(
        pareto_robustness_summary_df.sort_values(
            ["track_id", "pareto_membership_frequency"],
            ascending=[True, True],
            kind="mergesort",
        )
    )

if len(pareto_robustness_audit_df):
    display(
        pareto_robustness_audit_df.sort_values(
            ["track_id", "omitted_random_state"],
            kind="mergesort",
        )
    )


,model_id,track_id,reference_is_protocol_pareto,n_replicates,n_pareto_replicates,pareto_membership_frequency,front_stability_status,selection_influence
5,model_1b2c23b63e3e3a4c975a,E5,True,3,3,1.000000,stable_member,False
3,model_6bb2767778c7db4dff46,E6,False,3,0,0.000000,stable_non_member,False
1,model_0a7e299ead39b6a20953,E6,False,3,1,0.333333,membership_sensitive_to_base_seed,False
0,model_0326bb57ce83ca64ed15,E6,True,3,2,0.666667,membership_sensitive_to_base_seed,False
2,model_23963a68a13219879027,E6,True,3,2,0.666667,membership_sensitive_to_base_seed,False
4,model_8ce785efa5b5327a9667,E6,True,3,3,1.000000,stable_member,False
7,model_54a8a3135519ad96284c,E8,True,3,0,0.000000,stable_non_member,False
8,model_71d461cea84570e8e875,E8,False,3,0,0.000000,stable_non_member,False
11,model_f662665e0f4a7a7c4783,E8,False,3,1,0.333333,membership_sensitive_to_base_seed,False
10,model_c8d347632e1452f6d9af,E8,True,3,2,0.666667,membership_sensitive_to_base_seed,False


,track_id,omitted_random_state,n_candidate_models,n_reference_pareto,n_replicate_pareto,pareto_jaccard_vs_reference,selection_influence
1,E5,0,1,1,1,1.00,False
4,E5,1,1,1,1,1.00,False
7,E5,2,1,1,1,1.00,False
0,E6,0,5,3,3,1.00,False
3,E6,1,5,3,3,1.00,False
6,E6,2,5,3,2,0.25,False
2,E8,0,6,4,3,0.75,False
5,E8,1,6,4,4,0.60,False
8,E8,2,6,4,2,0.50,False


### J3 — Sensibilité locale du verrou spatial 03C


In [17]:
spatial_sensitivity_plan_df = build_spatial_sensitivity_plan(
    spatial_lock,
)

if (
    bool(expcfg.SIMCA_ROBUSTNESS_RUN_SPATIAL_SENSITIVITY)
    and len(spatial_sensitivity_plan_df)
):
    spatial_sensitivity_metrics_df = evaluate_spatial_sensitivity(
        spatial_sensitivity_plan_df,
        robustness_validation_executions_df,
        robustness_seed_thresholds_df,
        robustness_pixel_predictions_df,
        image_db,
        spatial_lock,
        robustness_spatial_component_metrics_df,
    )
else:
    spatial_sensitivity_metrics_df = pd.DataFrame(
        columns=expcfg.SIMCA_ROBUSTNESS_SPATIAL_SENSITIVITY_COLUMNS
    )

for name, frame in {
    "spatial_sensitivity_plan": spatial_sensitivity_plan_df,
    "spatial_sensitivity_metrics": spatial_sensitivity_metrics_df,
}.items():
    assert_supporting_only(frame, name=name)

print("Spatial sensitivity variants:", len(spatial_sensitivity_plan_df))
print("Spatial sensitivity metric rows:", len(spatial_sensitivity_metrics_df))

if len(spatial_sensitivity_metrics_df):
    display(
        spatial_sensitivity_metrics_df.sort_values(
            ["track_id", "factor", "model_id", "random_state", "metric"],
            kind="mergesort",
        ).head(80)
    )


Spatial sensitivity variants: 12
Spatial sensitivity metric rows: 2160


,model_id,random_state,track_id,factor,alternative_spatial_candidate_id,metric,reference_value,alternative_value,delta,practical_tolerance,effect_status,directional_status,selection_influence
0,model_3ba4782ef74eb4daaa5d,0,E8,connectivity,spatial_e400c156d3117af6,component_precision,0.330065,0.320000,-1.006537e-02,0.05,within_practical_tolerance,practically_equivalent,False
1,model_3ba4782ef74eb4daaa5d,0,E8,connectivity,spatial_e400c156d3117af6,component_recall,1.000000,1.000000,0.000000e+00,0.05,within_practical_tolerance,practically_equivalent,False
2,model_3ba4782ef74eb4daaa5d,0,E8,connectivity,spatial_e400c156d3117af6,dice,0.912880,0.912880,1.893827e-08,0.03,within_practical_tolerance,practically_equivalent,False
3,model_3ba4782ef74eb4daaa5d,0,E8,connectivity,spatial_e400c156d3117af6,iou,0.839723,0.839723,-2.345224e-08,0.03,within_practical_tolerance,practically_equivalent,False
4,model_3ba4782ef74eb4daaa5d,0,E8,connectivity,spatial_e400c156d3117af6,merge_rate,0.000000,0.000000,0.000000e+00,0.05,within_practical_tolerance,practically_equivalent,False
5,model_3ba4782ef74eb4daaa5d,0,E8,connectivity,spatial_e400c156d3117af6,pixel_precision,0.846105,0.846105,-2.850000e-08,0.05,within_practical_tolerance,practically_equivalent,False
6,model_3ba4782ef74eb4daaa5d,0,E8,connectivity,spatial_e400c156d3117af6,pixel_recall,0.991098,0.991098,-4.244841e-09,0.05,within_practical_tolerance,practically_equivalent,False
7,model_3ba4782ef74eb4daaa5d,0,E8,connectivity,spatial_e400c156d3117af6,smallest_fragment_recall,1.000000,1.000000,0.000000e+00,0.05,within_practical_tolerance,practically_equivalent,False
8,model_3ba4782ef74eb4daaa5d,0,E8,connectivity,spatial_e400c156d3117af6,split_rate,0.339623,0.169811,-1.698113e-01,0.05,outside_practical_tolerance,alternative_better,False
54,model_3ba4782ef74eb4daaa5d,1,E8,connectivity,spatial_e400c156d3117af6,component_precision,0.325658,0.302752,-2.290561e-02,0.05,within_practical_tolerance,practically_equivalent,False


### J4 — Sensibilité au choix des folds de calibration


In [18]:
(
    fold_sensitivity_plan_df,
    fold_sensitivity_assignments_df,
) = build_calibration_fold_sensitivity(
    folds_df,
    candidate_random_states=(
        expcfg.SIMCA_ROBUSTNESS_FOLD_SENSITIVITY_RANDOM_STATES
    ),
    max_unique_alternatives=(
        expcfg.SIMCA_ROBUSTNESS_FOLD_SENSITIVITY_MAX_UNIQUE_ALTERNATIVES
    ),
)

fold_sensitivity_thresholds_df = pd.DataFrame(
    columns=expcfg.SIMCA_ROBUSTNESS_FOLD_SENSITIVITY_THRESHOLD_COLUMNS
)
fold_sensitivity_metrics_df = pd.DataFrame(
    columns=expcfg.SIMCA_ROBUSTNESS_FOLD_SENSITIVITY_METRIC_COLUMNS
)
fold_sensitivity_decisions_df = pd.DataFrame(
    columns=expcfg.SIMCA_ROBUSTNESS_FOLD_SENSITIVITY_DECISION_COLUMNS
)
fold_sensitivity_technical_events_df = pd.DataFrame(
    columns=expcfg.SIMCA_ROBUSTNESS_FOLD_SENSITIVITY_TECHNICAL_EVENT_COLUMNS
)

estimable_fold_sensitivity = (
    bool(expcfg.SIMCA_ROBUSTNESS_RUN_FOLD_SENSITIVITY)
    and fold_sensitivity_plan_df["plan_status"]
    .astype(str)
    .eq("estimable_unique_valid_partition")
    .any()
)

if estimable_fold_sensitivity:
    fold_sensitivity_outputs = evaluate_calibration_fold_sensitivity(
        fold_plan=fold_sensitivity_plan_df,
        alternative_fold_assignments=fold_sensitivity_assignments_df,
        object_db=object_db,
        wavelengths=wavelengths,
        model_catalog=model_catalog_df,
        pareto_candidates=pareto_candidates_df,
        validation_executions=validation_executions_df,
        frozen_selected_thresholds=selected_thresholds_df,
        reference_validation_metrics=validation_metrics_df,
        object_predictions=validation_object_predictions_df,
        pixel_predictions=validation_pixel_predictions_df,
        checkpoint_root=FOLD_SENSITIVITY_CHECKPOINT_DIR,
        protocol_hash=PROTOCOL_HASH,
        checkpoint_context_base={
            "pca_selection_fingerprint": str(
                manifest_03b["pca_selection_fingerprint"]
            ),
            "track_contract_hash": str(
                manifest_03b["track_contract_hash"]
            ),
        },
    )
    fold_sensitivity_thresholds_df = fold_sensitivity_outputs["thresholds"]
    fold_sensitivity_metrics_df = fold_sensitivity_outputs["metrics"]
    fold_sensitivity_decisions_df = fold_sensitivity_outputs["decisions"]
    fold_sensitivity_technical_events_df = fold_sensitivity_outputs[
        "technical_events"
    ]

for name, frame in {
    "fold_sensitivity_plan": fold_sensitivity_plan_df,
    "fold_sensitivity_metrics": fold_sensitivity_metrics_df,
    "fold_sensitivity_decisions": fold_sensitivity_decisions_df,
}.items():
    assert_supporting_only(frame, name=name)

display(fold_sensitivity_plan_df)

if not estimable_fold_sensitivity:
    print(
        "No scientifically distinct admissible calibration-fold partition "
        "was estimable. This is an explicit limitation, not evidence of "
        "robustness."
    )
else:
    print("Alternative fold partitions:", len(fold_sensitivity_assignments_df["alternative_partition_sha256"].unique()))
    print("Fold-sensitivity metric rows:", len(fold_sensitivity_metrics_df))
    print("Fold-sensitivity decision rows:", len(fold_sensitivity_decisions_df))


,generator_random_state,reference_partition_sha256,alternative_partition_sha256,n_source_images,n_folds,coverage_complete,plan_status,selection_influence
0,<NA>,f4248ec7eb89b862c5087c7bc082dc9a8c589d915ac41e...,,4,2,True,not_estimable_no_alternative_valid_group_split,False


No scientifically distinct admissible calibration-fold partition was estimable. This is an explicit limitation, not evidence of robustness.


## K — Revue pré-batch4, ablations et diagnostics descriptifs

La décision de revue 05 repose uniquement sur le **Pareto officiel**, les guardrails amont et la robustesse multi-seeds configurée. Les diagnostics supporting de J ne sont pas injectés dans `build_robustness_review_guardrails`.

Les ablations sont également supporting-only : un modèle Pareto est comparé à ses contre-factuels exacts parmi les modèles réellement évalués en 04C du même track, avec effets appariés par seed lorsque les exécutions communes existent.


In [21]:
review_guardrails_df, track_review_df, pure_test_candidates_df = (
    build_robustness_review_guardrails(
        pareto_candidates_df,
        stability_summary_df,
    )
)

# Filename retained for downstream compatibility; semantics = review table only.
# No score, rank or second selection pass is created.
track_scoring_flags_df = track_review_df.copy()

ablation_plan_df = build_robustness_ablation_plan(
    model_catalog_df,
    pareto_candidates_df,
    evaluated_models=selection_units_df,
)
ablation_diagnostics_df = build_ablation_diagnostics(
    ablation_plan_df,
    selection_members_df,
)
ablation_coverage_df = build_ablation_coverage(
    ablation_plan_df,
    pareto_candidates_df,
)

statistical_uncertainty_df = build_descriptive_uncertainty_envelope(
    pure_test_candidates_df,
    combined_validation_metrics_df,
)
risk_coverage_df = build_risk_coverage_curves(
    pure_test_candidates_df,
    robustness_validation_executions_df,
    robustness_seed_thresholds_df,
    robustness_object_predictions_df,
    robustness_pixel_predictions_df,
    coverage_grid=expcfg.SIMCA_ROBUSTNESS_RISK_COVERAGE_GRID,
)

for name, frame in {
    "ablation_plan": ablation_plan_df,
    "ablation_diagnostics": ablation_diagnostics_df,
    "ablation_coverage": ablation_coverage_df,
    "statistical_uncertainty": statistical_uncertainty_df,
    "risk_coverage": risk_coverage_df,
}.items():
    assert_supporting_only(frame, name=name)

review_summary = (
    track_review_df
    .groupby(["track_id", "review_status"], as_index=False)
    .size()
    .rename(columns={"size": "n_models"})
    .sort_values(["track_id", "review_status"], kind="mergesort")
)

display(review_summary)
display(ablation_coverage_df)
print(
    "Candidates frozen for later pure-test evaluation:",
    len(pure_test_candidates_df),
)


,track_id,review_status,n_models
0,E1,excluded_missing_seed,2
1,E2,excluded_04c_guardrail,1
2,E2,excluded_missing_seed,7
3,E3,diagnostic_only,1
4,E4,diagnostic_only,1
5,E5,eligible_for_pure_test,1
6,E6,eligible_with_warning,2
7,E6,excluded_04c_guardrail,7
8,E6,excluded_missing_seed,5
9,E6,excluded_unstable,1


,track_id,factor,n_reference_pareto_models,n_exact_counterfactual_pairs,n_reference_models_with_counterfactual,reference_coverage_rate,coverage_status,selection_influence
0,E1,matrix_representation,0,0,0,NaN,not_applicable_no_pareto_reference,False
1,E1,m,0,0,0,NaN,not_applicable_no_pareto_reference,False
2,E1,balanced_pixel_strategy,0,0,0,NaN,not_applicable_no_pareto_reference,False
3,E1,preprocessing,0,0,0,NaN,not_applicable_no_pareto_reference,False
4,E1,rule_variant,0,0,0,NaN,not_applicable_no_pareto_reference,False
5,E1,n_components,0,0,0,NaN,not_applicable_no_pareto_reference,False
6,E1,sg_window_length,0,0,0,NaN,not_applicable_no_pareto_reference,False
7,E1,position_dilation_radius,0,0,0,NaN,not_applicable_no_pareto_reference,False
8,E2,matrix_representation,0,0,0,NaN,not_applicable_no_pareto_reference,False
9,E2,m,0,0,0,NaN,not_applicable_no_pareto_reference,False


Candidates frozen for later pure-test evaluation: 4


## L — Persistance sous contrats centraux et verrou pré-batch4

Chaque table est écrite avec son schéma déclaré dans `experiment_config.PIPELINE_TABLE_CONTRACTS`. Les diagnostics supporting sont persistés pour l’audit, mais leur `selection_influence` reste faux.

`pure_test_candidate_registry.parquet` est la frontière gelée transmise à la suite. Le verrou enregistre à la fois le hash du fichier et le hash canonique du DataFrame relu.


In [22]:
tables_to_save = {
    "selection_units": selection_units_df,
    "selection_members": selection_members_df,
    "pareto_candidates": pareto_candidates_df,
    "pareto_audit": pareto_audit_df,
    "seed_executions": robustness_seed_executions_df,
    "seed_thresholds": robustness_seed_thresholds_df,
    "seed_metrics": robustness_seed_metrics_df,
    "stability_summary": stability_summary_df,
    "seed_disagreement": seed_disagreement_df,
    "threshold_sensitivity_plan": threshold_sensitivity_plan_df,
    "threshold_sensitivity_metrics": threshold_sensitivity_metrics_df,
    "threshold_sensitivity_decisions": threshold_sensitivity_decisions_df,
    "threshold_stability": threshold_stability_df,
    "source_image_influence": source_image_influence_df,
    "fold_sensitivity_plan": fold_sensitivity_plan_df,
    "fold_sensitivity_assignments": fold_sensitivity_assignments_df,
    "fold_sensitivity_thresholds": fold_sensitivity_thresholds_df,
    "fold_sensitivity_metrics": fold_sensitivity_metrics_df,
    "fold_sensitivity_decisions": fold_sensitivity_decisions_df,
    "fold_sensitivity_technical_events": fold_sensitivity_technical_events_df,
    "pareto_robustness_replicates": pareto_robustness_replicates_df,
    "pareto_robustness_summary": pareto_robustness_summary_df,
    "pareto_robustness_audit": pareto_robustness_audit_df,
    "spatial_sensitivity_plan": spatial_sensitivity_plan_df,
    "spatial_sensitivity_metrics": spatial_sensitivity_metrics_df,
    "ablation_plan": ablation_plan_df,
    "ablation_diagnostics": ablation_diagnostics_df,
    "ablation_coverage": ablation_coverage_df,
    "statistical_uncertainty": statistical_uncertainty_df,
    "risk_coverage": risk_coverage_df,
    "review_guardrails": review_guardrails_df,
    "track_scoring_flags": track_scoring_flags_df,
    "pure_test_candidates": pure_test_candidates_df,
}

for key, table in tables_to_save.items():
    write_simca_table(
        table,
        OUTPUT_PATHS[key],
        include_remaining=False,
        drop_all_na=False,
        validate_keys=True,
        require_all_columns=True,
    )

persisted_tables = {
    key: read_simca_table(
        OUTPUT_PATHS[key],
        required=True,
        include_remaining=False,
        drop_all_na=False,
        validate_keys=True,
        require_all_columns=True,
    )
    for key in tables_to_save
}
schema_manifest_df = build_schema_manifest(persisted_tables)

OUTPUT_SHA256 = {
    key: sha256_file(OUTPUT_PATHS[key])
    for key in tables_to_save
}
UPSTREAM_SHA256 = {
    **{
        name: sha256_file(path)
        for name, path in EARLY_LINEAGE_PATHS.items()
    },
    "03B.checkpoint_manifest": sha256_file(INPUT_03B_MANIFEST),
    **{
        f"03B.{key}": sha256_file(path)
        for key, path in INPUT_03B.items()
    },
    **{
        f"03C.{key}": sha256_file(path)
        for key, path in INPUT_03C.items()
    },
    **{
        f"04A.{key}": sha256_file(path)
        for key, path in INPUT_04A.items()
    },
    **{
        f"04C.{key}": sha256_file(path)
        for key, path in INPUT_04C.items()
    },
}
if INPUT_04B_AUDIT.is_file():
    UPSTREAM_SHA256["04B.audit_manifest"] = sha256_file(
        INPUT_04B_AUDIT
    )

track_counts = (
    pareto_candidates_df
    .groupby("track_id", as_index=False)
    .agg(
        n_base_models=("model_id", "nunique"),
        n_protocol_pareto=("is_protocol_pareto", "sum"),
    )
    .merge(
        track_review_df.groupby("track_id", as_index=False).agg(
            n_reviewed_models=("model_id", "nunique"),
            n_hard_exclusions=("hard_exclusion", "sum"),
        ),
        on="track_id",
        how="left",
        validate="one_to_one",
    )
)

protocol_payload = {
    "notebook": "05_simca_validation_robustness",
    "results_tag": RESULTS_TAG,
    "protocol_version": str(expcfg.PROTOCOL_VERSION),
    "schema_version": str(expcfg.RESULTS_SCHEMA_VERSION),
    "global_protocol_hash": PROTOCOL_HASH,
    "contract_version": str(
        expcfg.SIMCA_ROBUSTNESS_CONTRACT_VERSION
    ),
    "contract_role": str(expcfg.SIMCA_ROBUSTNESS_CONTRACT_ROLE),
    "child_contract": robustness_contract_payload(),
    "child_contract_hash": ROBUSTNESS_CONTRACT_HASH,
    "scientific_identity": "model_id",
    "execution_identity": ["model_id", "random_state"],
    "decision_policy_identity": [
        "model_id",
        "random_state",
        "decision_scope",
    ],
    "pareto_scope": "track_id",
    "pareto_panel": "04C_base_random_states_only",
    "pareto_seed_aggregation": str(
        expcfg.SIMCA_ROBUSTNESS_PARETO_SEED_AGGREGATION
    ),
    "pareto_base_random_states": list(
        map(int, expcfg.SIMCA_ROBUSTNESS_BASE_RANDOM_STATES)
    ),
    "pareto_recomputed_after_additional_seeds": False,
    "additional_seed_random_states": list(
        map(int, expcfg.SIMCA_ROBUSTNESS_ADDITIONAL_RANDOM_STATES)
    ),
    "additional_seed_robustness_enabled": bool(
        expcfg.SIMCA_ROBUSTNESS_RUN_ADDITIONAL_SEEDS
    ),
    "threshold_policy_reselected_for_additional_seeds": False,
    "numeric_thresholds_recalibrated_per_additional_seed": True,
    "additional_seed_calibration_batches": list(
        map(int, expcfg.INTERNAL_CALIBRATION_BATCHES)
    ),
    "additional_seed_validation_batches": list(
        map(int, expcfg.SIMCA_CONCAT_REFIT_PROJECTION_BATCHES)
    ),
    "stability_registration_status": str(
        expcfg.SIMCA_ROBUSTNESS_STABILITY_REGISTRATION_STATUS
    ),
    "supporting_diagnostic_rule_version": str(
        expcfg.SIMCA_ROBUSTNESS_SUPPORTING_DIAGNOSTIC_RULE_VERSION
    ),
    "supporting_diagnostic_registration_status": str(
        expcfg.SIMCA_ROBUSTNESS_SUPPORTING_DIAGNOSTIC_REGISTRATION_STATUS
    ),
    "uncertainty_summary_semantics": str(
        expcfg.SIMCA_ROBUSTNESS_UNCERTAINTY_SUMMARY_SEMANTICS
    ),
    "threshold_sensitivity_selection_influence": False,
    "threshold_stability_selection_influence": False,
    "source_image_influence_selection_influence": False,
    "fold_sensitivity_selection_influence": False,
    "pareto_front_sensitivity_selection_influence": False,
    "spatial_sensitivity_selection_influence": False,
    "ablation_selection_influence": False,
    "risk_coverage_selection_influence": False,
    "official_pareto_recomputed_by_sensitivity_analyses": False,
    "official_spatial_lock_modified_by_sensitivity_analyses": False,
    "batch4_loaded": False,
    "final_model_selection_performed": False,
    "cross_track_selection_or_ranking": False,
    "weighted_score_used": False,
    "candidate_registry_frozen_before_batch4": True,
    "intentionally_not_loaded_upstream": {
        "01B_spatial_ground_truth": (
            "contains batch4/test spatial truth; lineage is carried by "
            "downstream manifests but the artefacts are not opened in "
            "notebook 05"
        )
    },
    "04c_execution_registry_sha256": execution_hash,
    "04c_selected_thresholds_sha256": threshold_hash,
    "04c_validation_plan_hash": str(
        validation_protocol["validation_plan_hash"]
    ),
    "04c_validation_evaluation_rule_hash": str(
        validation_protocol["validation_evaluation_rule_hash"]
    ),
    "additional_execution_registry_sha256": (
        ADDITIONAL_EXECUTION_REGISTRY_HASH
    ),
    "additional_validation_plan_hash": (
        ADDITIONAL_VALIDATION_PLAN_HASH
    ),
    "database_sha256": DATABASE_SHA256,
    "additional_seed_calibration_checkpoint_dir": (
        str(additional_calibration_outputs["checkpoint_run_dir"])
        if additional_calibration_outputs is not None
        else None
    ),
    "additional_seed_validation_checkpoint_dir": (
        None
        if ADDITIONAL_VALIDATION_CHECKPOINT_RUN_DIR is None
        else str(ADDITIONAL_VALIDATION_CHECKPOINT_RUN_DIR)
    ),
    "fold_sensitivity_checkpoint_dir": (
        str(FOLD_SENSITIVITY_CHECKPOINT_DIR)
        if estimable_fold_sensitivity
        else None
    ),
    "track_counts": track_counts.to_dict(orient="records"),
    "input_sha256": UPSTREAM_SHA256,
    "output_sha256": OUTPUT_SHA256,
}
OUTPUT_PATHS["protocol"].write_text(
    json.dumps(
        protocol_payload,
        indent=2,
        sort_keys=True,
        ensure_ascii=False,
        default=str,
    )
    + "\n",
    encoding="utf-8",
)

persisted_candidates_df = read_simca_table(
    OUTPUT_PATHS["pure_test_candidates"],
    required=True,
    include_remaining=False,
    drop_all_na=False,
    validate_keys=True,
    require_all_columns=True,
)

lock_payload = {
    "lock_status": "immutable_pre_batch4_candidate_registry",
    "notebook": "05_simca_validation_robustness",
    "contract_role": str(expcfg.SIMCA_ROBUSTNESS_CONTRACT_ROLE),
    "protocol_version": str(expcfg.PROTOCOL_VERSION),
    "global_protocol_hash": PROTOCOL_HASH,
    "child_contract_hash": ROBUSTNESS_CONTRACT_HASH,
    "batch4_loaded": False,
    "final_model_selection_performed": False,
    "scientific_identity": "model_id",
    "execution_identity": ["model_id", "random_state"],
    "pareto_scope": "track_id",
    "cross_track_selection_or_ranking": False,
    "weighted_score_used": False,
    "candidate_registry_frozen_before_batch4": True,
    "supporting_diagnostics_selection_influence": False,
    "uncertainty_summary_semantics": str(
        expcfg.SIMCA_ROBUSTNESS_UNCERTAINTY_SUMMARY_SEMANTICS
    ),
    "candidate_registry_file": str(
        OUTPUT_PATHS["pure_test_candidates"]
    ),
    "candidate_registry_file_sha256": sha256_file(
        OUTPUT_PATHS["pure_test_candidates"]
    ),
    "candidate_registry_dataframe_sha256": sha256_dataframe(
        persisted_candidates_df
    ),
    "review_protocol_sha256": sha256_file(OUTPUT_PATHS["protocol"]),
    "input_sha256": UPSTREAM_SHA256,
    "output_sha256": OUTPUT_SHA256,
}
lock_payload["lock_sha256"] = sha256_payload(lock_payload)
OUTPUT_PATHS["lock_manifest"].write_text(
    json.dumps(
        lock_payload,
        indent=2,
        sort_keys=True,
        ensure_ascii=False,
        default=str,
    )
    + "\n",
    encoding="utf-8",
)

print("Saved formal 05 outputs:", len(tables_to_save) + 2)
print(
    "Candidate registry dataframe SHA-256:",
    lock_payload["candidate_registry_dataframe_sha256"],
)
print("05 lock SHA-256:", lock_payload["lock_sha256"])
display(schema_manifest_df)


Saved formal 05 outputs: 35
Candidate registry dataframe SHA-256: 11ee71a4e7e4877fe736982ddf958b6ee217acfd1765d973035d9ed1d2cad73a
05 lock SHA-256: 15d8f62945441c0c0acd0e999cbc22a4379845dbb2f99484a8199debea9a32a2


,table_name,n_rows,n_columns,n_all_na_columns,n_suffix_columns,all_na_columns,suffix_columns
0,selection_units,38,452,20,0,"direct__worst_image__balanced_accuracy,direct_...",
1,selection_members,104,85,2,0,"worst_image__balanced_accuracy,worst_image__de...",
2,pareto_candidates,38,460,20,0,"direct__worst_image__balanced_accuracy,direct_...",
3,pareto_audit,76,10,0,0,,
4,seed_executions,80,4,0,0,,
5,seed_thresholds,120,8,1,0,vote_threshold,
6,seed_metrics,120,85,2,0,"worst_image__balanced_accuracy,worst_image__de...",
7,stability_summary,181,36,0,0,,
8,seed_disagreement,12,10,0,0,,
9,threshold_sensitivity_plan,460,10,0,0,,


## M — Contrôles finaux pour la suite et pour le notebook d’audit


In [23]:
# Formal read-back guarantees for downstream notebooks and the selection audit.
if set(
    persisted_tables["pareto_candidates"]["track_id"].astype(str)
) != expected_tracks:
    raise RuntimeError(
        "Persisted Pareto table no longer covers exactly E1-E8."
    )

if persisted_tables["pure_test_candidates"].duplicated(
    ["model_id", "track_id"]
).any():
    raise RuntimeError(
        "Persisted pre-batch4 candidate registry duplicates "
        "its natural key."
    )

all_scientific_columns = set().union(
    *(set(frame.columns) for frame in persisted_tables.values())
)
legacy_identifiers = {
    "calibration_id",
    "validation_candidate_id",
    "selected_config_id",
    "seed_id",
    "execution_id",
    "robustness_model_id",
    "ablation_id",
    "sensitivity_id",
    "fold_sensitivity_id",
    "spatial_sensitivity_id",
}
leaked = sorted(
    legacy_identifiers.intersection(all_scientific_columns)
)
if leaked:
    raise RuntimeError(
        "Legacy/surrogate identifiers leaked into notebook-05 "
        f"outputs: {leaked}"
    )

forbidden_scores = set(
    expcfg.ACTIVE_PROTOCOL_FORBIDDEN_SCORE_COLUMNS
)
score_leaks = sorted(
    forbidden_scores.intersection(all_scientific_columns)
)
if score_leaks:
    raise RuntimeError(
        "Forbidden score/rank columns leaked into notebook-05 "
        f"outputs: {score_leaks}"
    )

supporting_keys = (
    "threshold_sensitivity_plan",
    "threshold_sensitivity_metrics",
    "threshold_sensitivity_decisions",
    "threshold_stability",
    "source_image_influence",
    "fold_sensitivity_plan",
    "fold_sensitivity_metrics",
    "fold_sensitivity_decisions",
    "pareto_robustness_replicates",
    "pareto_robustness_summary",
    "pareto_robustness_audit",
    "spatial_sensitivity_plan",
    "spatial_sensitivity_metrics",
    "ablation_plan",
    "ablation_diagnostics",
    "ablation_coverage",
    "statistical_uncertainty",
    "risk_coverage",
)
for key in supporting_keys:
    assert_supporting_only(
        persisted_tables[key],
        name=f"persisted {key}",
    )

if len(persisted_tables["threshold_stability"]):
    if not persisted_tables["threshold_stability"][
        "policy_coordinates_invariant"
    ].fillna(False).astype(bool).all():
        raise RuntimeError(
            "Threshold-policy coordinates changed during robustness."
        )

persisted_protocol = json.loads(
    OUTPUT_PATHS["protocol"].read_text(encoding="utf-8")
)
persisted_lock = json.loads(
    OUTPUT_PATHS["lock_manifest"].read_text(encoding="utf-8")
)

if persisted_protocol["batch4_loaded"] is not False:
    raise RuntimeError(
        "Notebook-05 protocol reports batch4 exposure."
    )
if persisted_protocol["final_model_selection_performed"] is not False:
    raise RuntimeError(
        "Notebook-05 protocol reports a final selection."
    )
if persisted_protocol[
    "official_pareto_recomputed_by_sensitivity_analyses"
] is not False:
    raise RuntimeError(
        "Supporting diagnostics altered the official Pareto."
    )
if persisted_protocol[
    "official_spatial_lock_modified_by_sensitivity_analyses"
] is not False:
    raise RuntimeError(
        "Supporting diagnostics altered the official spatial lock."
    )
if persisted_protocol["uncertainty_summary_semantics"] != str(
    expcfg.SIMCA_ROBUSTNESS_UNCERTAINTY_SUMMARY_SEMANTICS
):
    raise RuntimeError(
        "Persisted uncertainty semantics differ from the child contract."
    )

if persisted_lock[
    "candidate_registry_dataframe_sha256"
] != sha256_dataframe(
    persisted_tables["pure_test_candidates"]
):
    raise RuntimeError(
        "Candidate registry dataframe hash changed after persistence."
    )

if persisted_lock["supporting_diagnostics_selection_influence"] is not False:
    raise RuntimeError(
        "The 05 lock incorrectly gives selection authority to "
        "supporting diagnostics."
    )

print("Notebook 05 contract checks passed.")
print(
    "Downstream candidate registry:",
    OUTPUT_PATHS["pure_test_candidates"],
)
print("Audit protocol:", OUTPUT_PATHS["protocol"])
print("Audit lock:", OUTPUT_PATHS["lock_manifest"])


Notebook 05 contract checks passed.
Downstream candidate registry: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\05_simca_validation_robustness_8tracks_v5_px_qc_v1\pure_test_candidate_registry.parquet
Audit protocol: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\05_simca_validation_robustness_8tracks_v5_px_qc_v1\robustness_review_protocol.json
Audit lock: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\05_simca_validation_robustness_8tracks_v5_px_qc_v1\robustness_review_lock.json
